# 25882 AI-powered Investment and Risk Management — Assessment 1: Empirical Assignment

**Group members (name, student ID):**
- TODO: Name 1, Student ID 1
- TODO: Name 2, Student ID 2 *(delete this line if working alone)*


## Part B choices

We attempt **B1 (Portfolio optimisation)**, from Category B, as **Extension 1**, and
**A3 (Tail risk and stress testing)**, from Category A, as **Extension 2**. This
satisfies the requirement that the two options come from different categories (B and A)
with at least one from Category A or B (both are).

The two are deliberately linked: Extension 1 builds minimum-variance, maximum-Sharpe and
risk-parity portfolios; Extension 2 stress-tests those same portfolios against the
fat-tailed, correlation-spiking behaviour already documented in A3 (core) -- including an
EVT-based tail estimate compared directly against the historical and parametric VaR/ES
computed there.

Note on labelling: the assignment's Category A option 3 is *also* called "A3", which
coincidentally collides with this notebook's own core section A3 (Baseline risk report).
Per the brief's own heading convention, Part B options are headed `## Extension 1` /
`## Extension 2`, not a re-used `## A3` -- so there is no actual heading collision, just a
naming coincidence worth flagging so it is not mistaken for an error.

In [ ]:
import importlib.util
import subprocess
import sys


def _ensure_installed(packages):
    """
    Cheap safety net for `Kernel -> Restart & Run All` on a machine where the
    notebook's kernel does not match the environment `pip install -r
    requirements.txt` was run into (a common Jupyter/conda mix-up). Checks
    each package's presence in *this* kernel only -- not its version -- and
    installs only what is actually missing, into `sys.executable` so it
    cannot disagree with itself. A no-op, and silent, when everything is
    already available -- the expected case in a correctly set-up environment.
    """
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if missing:
        print(f"Installing missing packages into this kernel: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


_ensure_installed(["numpy", "pandas", "yfinance"])

import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data_cache_bhp")  # separate from the original notebook's data_cache/ so the two never share (and silently collide on) the same cache files
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"Notebook environment set up. Random seed fixed at {RANDOM_SEED}.")


## A1 — Universe selection and data acquisition

### Asset universe and justification

| Ticker | Name | Sector | Exchange | Currency |
|---|---|---|---|---|
| AAPL | Apple Inc. | Information Technology | NASDAQ | USD |
| JPM | JPMorgan Chase & Co. | Financials | NYSE | USD |
| XOM | ExxonMobil Corp. | Energy | NYSE | USD |
| JNJ | Johnson & Johnson | Health Care | NYSE | USD |
| BHP.AX | BHP Group Limited | Materials (Mining) | Australian Securities Exchange (ASX) | AUD |

We chose four large-cap US stocks spanning four distinct GICS sectors (technology, financials, energy,
healthcare) plus one Australian-listed stock (BHP Group, BHP.AX) to satisfy two goals at once. First, five
distinct sectors avoid the degenerate, near-perfectly-correlated universe a single-sector selection
would produce, which the brief warns would flatten the diversification and optimisation results in
Part B -- BHP's materials/mining sector is driven by a very different macro factor (global commodity
demand, particularly from China) than any of the other four. Second, BHP is listed on the Australian
Securities Exchange and trades in AUD, giving us a genuine cross-currency, cross-trading-calendar asset
-- required by the brief, and necessary to make the currency-alignment and holiday-calendar corrections
in Section A2 non-vacuous. A different choice -- five US large-caps in USD, say -- would have prevented
us from ever observing an FX or calendar-alignment effect at all, and would have understated how much
diversification the equal-weight benchmark in A4 can actually deliver.

In [ ]:
# --- Universe and sample period ---
TICKERS = ["AAPL", "JPM", "XOM", "JNJ", "BHP.AX"]

# Fixed, hard-coded date range (not "today") so the notebook is exactly
# reproducible on re-run: the price data requested does not depend on when
# the notebook happens to be executed. This spans just over 9 years, safely
# above the 8-year minimum.
PRICE_START = "2016-01-01"
PRICE_END = "2025-01-01"

# Recorded once, manually, at the time of the first successful live download
# below -- this is metadata about *when we pulled the data*, not a parameter
# that should silently change what gets downloaded on a later re-run.
DOWNLOAD_DATE_RECORDED = "2026-09-15"  # TODO: update to the actual date you first run this successfully

print(f"Universe: {TICKERS}")
print(f"Requested date range: {PRICE_START} to {PRICE_END}")


In [ ]:
def load_price_panel(tickers, start, end, cache_dir=DATA_DIR, force_download=False):
    """
    Download daily unadjusted and dividend/split-adjusted close prices for
    `tickers` between `start` and `end`, and return them as two clean, wide,
    date-aligned DataFrames (one column per ticker).

    Used by every later section of this notebook -- built once here, reused
    throughout (including for the FX and risk-free single-ticker series in
    A2), per the assignment's instruction not to re-implement this.

    Behaviour
    ---------
    * yfinance now defaults to auto_adjust=True, which returns only an
      adjusted Close and discards the raw price. We call it with
      auto_adjust=False explicitly so both series are available, because
      Section A2 needs both.
    * yfinance returns MultiIndex (ticker, field) columns for a multi-ticker
      request, but sometimes returns FLAT columns for a single-ticker
      request even with group_by="ticker" -- both shapes are handled.
    * A ticker that returns nothing, or returns an all-NaN Close column, is
      dropped and reported -- *before* any row-wise NaN handling. Dropping
      NaN rows first would let one broken ticker delete the entire sample.
    * On success, the two wide panels are cached to `cache_dir` as CSV files.
      On a later call (or a later notebook re-run) with the cache already
      present, the function reads the cache directly and does not touch the
      network at all.
    * If a live download raises (network error, rate limit, vendor outage --
      "which happens", per the brief) the function falls back to the local
      cache if one exists, so the notebook degrades gracefully rather than
      crashing on a clean re-run at marking time.

    Parameters
    ----------
    tickers : list of str
    start, end : str, "YYYY-MM-DD"
    cache_dir : Path
    force_download : bool
        Skip a fresh-cache check and always attempt a live download first
        (still falls back to the cache on failure).

    Returns
    -------
    raw_close : DataFrame, date-indexed, one column per surviving ticker
    adj_close : DataFrame, date-indexed, one column per surviving ticker
    log : dict
        Provenance metadata: source, timestamp, tickers requested vs
        returned, yfinance version.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    raw_path = cache_dir / "raw_close.csv"
    adj_path = cache_dir / "adj_close.csv"

    if not force_download and raw_path.exists() and adj_path.exists():
        raw_close = pd.read_csv(raw_path, index_col=0, parse_dates=True)
        adj_close = pd.read_csv(adj_path, index_col=0, parse_dates=True)
        cached_tickers, requested_tickers = set(raw_close.columns), set(tickers)
        if cached_tickers == requested_tickers:
            log = {
                "source": "local cache",
                "path": str(cache_dir),
                "tickers": list(raw_close.columns),
            }
            print(f"Loaded cached panel from {cache_dir}/ "
                  f"(delete raw_close.csv / adj_close.csv there to force a fresh download).")
            return raw_close, adj_close, log
        else:
            # A stale or mismatched cache is worse than no cache: it looks like a
            # normal cache hit and returns the wrong tickers' data silently. This
            # happens in practice when two notebooks share the same relative
            # cache_dir (e.g. a copy of this notebook with a different TICKERS
            # list run in the same folder) -- caught only by explicitly checking
            # the cached columns against what was actually requested, not by
            # trusting "the files exist" as a sufficient cache-hit condition.
            print(f"Cache at {cache_dir}/ contains {sorted(cached_tickers)}, which does not "
                  f"match the requested tickers {sorted(requested_tickers)} -- ignoring this "
                  "stale/mismatched cache and attempting a fresh download instead.")

    try:
        data = yf.download(tickers, start=start, end=end, auto_adjust=False,
                            group_by="ticker", progress=False, threads=True)
        if data.empty:
            raise ValueError("yfinance returned an empty frame for every ticker")

        raw_cols, adj_cols = {}, {}
        if isinstance(data.columns, pd.MultiIndex):
            for t in tickers:
                try:
                    sub = data[t]
                except KeyError:
                    print(f"  WARNING: no data at all returned for {t}; dropping from panel")
                    continue
                if sub["Close"].dropna().empty:
                    print(f"  WARNING: {t} returned an all-NaN Close column; dropping from panel")
                    continue
                raw_cols[t] = sub["Close"]
                adj_cols[t] = sub["Adj Close"]
        else:
            # Flat-column response: only valid for a single requested ticker.
            if len(tickers) != 1:
                raise ValueError(
                    f"Expected MultiIndex columns for {len(tickers)} tickers, got flat "
                    f"columns {list(data.columns)}"
                )
            t = tickers[0]
            if data["Close"].dropna().empty:
                print(f"  WARNING: {t} returned an all-NaN Close column; dropping from panel")
            else:
                raw_cols[t] = data["Close"]
                adj_cols[t] = data["Adj Close"]

        if not raw_cols:
            raise ValueError("Every requested ticker came back empty or all-NaN")

        raw_close = pd.DataFrame(raw_cols).sort_index()
        adj_close = pd.DataFrame(adj_cols).sort_index()

        raw_close.to_csv(raw_path)
        adj_close.to_csv(adj_path)

        log = {
            "source": "yfinance (live download)",
            "download_timestamp": datetime.now().isoformat(timespec="seconds"),
            "yfinance_version": getattr(yf, "__version__", "unknown"),
            "requested_tickers": list(tickers),
            "returned_tickers": list(raw_close.columns),
            "requested_start": start,
            "requested_end": end,
        }
        print(f"Downloaded fresh data and cached it to {cache_dir}/.")
        return raw_close, adj_close, log

    except Exception as exc:
        print(f"Live download failed ({exc!r}).")
        if raw_path.exists() and adj_path.exists():
            print("Falling back to the previously cached local files so the "
                  "notebook can still run end to end.")
            raw_close = pd.read_csv(raw_path, index_col=0, parse_dates=True)
            adj_close = pd.read_csv(adj_path, index_col=0, parse_dates=True)
            log = {
                "source": "local cache (after a failed live download)",
                "path": str(cache_dir),
            }
            return raw_close, adj_close, log
        raise RuntimeError(
            f"No live data available and no local cache found at {cache_dir}/. "
            "Cannot proceed -- see the note on reproducibility in the assignment brief."
        ) from exc


In [ ]:
raw_close, adj_close, download_log = load_price_panel(TICKERS, PRICE_START, PRICE_END)

print("\nDownload log:")
for k, v in download_log.items():
    print(f"  {k}: {v}")


In [ ]:
def summarize_panel(price_df, label="", min_years=8):
    """
    Report, per asset: first date, last date, observation count, and years
    spanned -- and flag any asset whose history falls short of `min_years`.

    This is the "inspect what arrived before using it" step: it does not
    silently trust the download, it shows the evidence.
    """
    summary = pd.DataFrame({
        "first_date": price_df.apply(lambda s: s.dropna().index.min()),
        "last_date": price_df.apply(lambda s: s.dropna().index.max()),
        "n_obs": price_df.count(),
    })
    summary["years_span"] = (summary["last_date"] - summary["first_date"]).dt.days / 365.25

    print(f"--- {label} ---")
    display(summary)

    short = summary[summary["years_span"] < min_years]
    if not short.empty:
        print(f"WARNING: history shorter than {min_years} years for: {list(short.index)}")
    else:
        print(f"All assets meet the {min_years}-year minimum.")
    return summary


raw_summary = summarize_panel(raw_close, label="Raw close -- inspection", min_years=8)
adj_summary = summarize_panel(adj_close, label="Adjusted close -- inspection", min_years=8)


In [ ]:
ASSET_METADATA = {
    "AAPL":   {"name": "Apple Inc.",            "sector": "Information Technology",
               "exchange": "NASDAQ", "currency": "USD"},
    "JPM":    {"name": "JPMorgan Chase & Co.",  "sector": "Financials",
               "exchange": "NYSE",   "currency": "USD"},
    "XOM":    {"name": "ExxonMobil Corp.",      "sector": "Energy",
               "exchange": "NYSE",   "currency": "USD"},
    "JNJ":    {"name": "Johnson & Johnson",     "sector": "Health Care",
               "exchange": "NYSE",   "currency": "USD"},
    "BHP.AX": {"name": "BHP Group Limited",     "sector": "Materials",
               "exchange": "Australian Securities Exchange (ASX)", "currency": "AUD"},
}

asset_summary = (
    pd.DataFrame(ASSET_METADATA).T
    .join(raw_summary[["first_date", "last_date", "n_obs", "years_span"]])
)
asset_summary.index.name = "ticker"
asset_summary["years_span"] = asset_summary["years_span"].round(3)

display(asset_summary)

n_obs_gap = asset_summary["n_obs"].max() - asset_summary["n_obs"].min()
if n_obs_gap > 0:
    shortest = asset_summary["n_obs"].idxmin()
    print(f"\n{shortest} has {n_obs_gap} fewer trading-day observations than the fullest "
          f"series over the same nominal date range -- direct evidence that its exchange "
          f"does not share a trading calendar with the US tickers.")


In [ ]:
def load_ohlcv_preview(tickers, start, end, cache_dir=DATA_DIR / "ohlcv_preview",
                       force_download=False):
    """
    Download the FULL OHLCV panel (Open, High, Low, Close, Adj Close, Volume)
    purely for visual inspection of what the vendor actually served.

    This is display-only and is NOT what any later section computes from --
    every later section works from load_price_panel()'s raw_close/adj_close,
    which deliberately keeps only Close and Adj Close (A2 is specifically
    about auditing those two series). Same cache-then-live-then-graceful-
    skip discipline as the main loader, but a failure here does not raise:
    it is a display convenience, not something the rest of the notebook
    depends on.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / "ohlcv_long.csv"

    if not force_download and cache_path.exists():
        return pd.read_csv(cache_path, parse_dates=["date"])

    try:
        data = yf.download(tickers, start=start, end=end, auto_adjust=False,
                            group_by="ticker", progress=False, threads=True)
        if data.empty:
            raise ValueError("yfinance returned an empty frame")

        frames = []
        if isinstance(data.columns, pd.MultiIndex):
            for t in tickers:
                if t not in data.columns.get_level_values(0):
                    continue
                sub = data[t].reset_index().rename(columns={"Date": "date"})
                sub["ticker"] = t
                frames.append(sub)
        else:
            sub = data.reset_index().rename(columns={"Date": "date"})
            sub["ticker"] = tickers[0]
            frames.append(sub)

        long = pd.concat(frames, ignore_index=True)
        long = long[["date", "ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"]]
        long.to_csv(cache_path, index=False)
        return long

    except Exception as exc:
        print(f"OHLCV preview download failed ({exc!r}); this cell is display-only, "
              "skipping it gracefully -- it does not affect any other section.")
        return None


ohlcv_long = load_ohlcv_preview(TICKERS, PRICE_START, PRICE_END)

if ohlcv_long is not None:
    print("Sample of the raw downloaded data (first 3 rows per ticker):")
    display(ohlcv_long.groupby("ticker").head(3).set_index(["ticker", "date"]))

    print("\nDescriptive summary per ticker, full sample (Open/High/Low/Close/Adj Close/Volume):")
    ohlcv_stats = (
        ohlcv_long.groupby("ticker")[["Open", "High", "Low", "Close", "Adj Close", "Volume"]]
        .agg(["mean", "min", "max"])
        .round(2)
    )
    display(ohlcv_stats)


### A1 discussion

TODO once run with live data: comment on the actual date ranges and observation counts
returned -- in particular, whether BHP.AX's calendar (ASX trading days) gives a different
`n_obs` from the US tickers even over the same nominal date range. Australian public
holidays such as Australia Day, ANZAC Day, the King's Birthday and Melbourne Cup Day do
not coincide with the US holiday calendar (and the reverse also happens -- e.g. the ASX
trades as normal on US Thanksgiving), so a nonzero gap here is expected; report the exact
size of the gap and note whether any ticker's history fell short of the 8-year
requirement.

## A2 — Data and convention audit: the correction waterfall

We build a deliberately careless baseline, then apply one correction at a time -- each on
top of everything already corrected before it -- recomputing annualised return, annualised
volatility, Sharpe ratio and maximum drawdown after every step. The portfolio construction
method (equal weight, no explicit rebalancing logic beyond what `pct_change` implies) is
held fixed throughout, so the *only* thing changing row to row is the data/convention
correction under test.


In [ ]:
def equal_weight_portfolio_returns(price_panel, return_type="simple"):
    """
    Equal-weight (1/N), implicitly-daily-rebalanced SIMPLE portfolio return
    -- the weighted average of each asset's own simple return, which is the
    exact quantity for a dollar-weighted rebalanced portfolio.

    return_type="log" does NOT average individual assets' own log returns:
    mean(log(1+r_i)) != log(1 + mean(r_i)) whenever the r_i differ (Jensen's
    inequality on the concave log), so that would silently describe a
    DIFFERENT, not-quite-real portfolio rather than this one on a log scale
    -- confirmed on a synthetic panel with an injected single-asset shock,
    where that mistake alone moved max drawdown from -16.96% to -20.55%
    purely as an averaging artifact, nothing to do with simple-vs-log.
    Instead we take log1p of the portfolio's own simple return: exactly
    this portfolio, expressed on a log scale. The wealth path (and
    therefore drawdown) is then identical to the simple-return version by
    construction; only the annualised-return statistic differs, which is
    the arithmetic-vs-geometric-mean point Step 8 is actually about.

    If a column is NaN on a given date (e.g. a holiday gap we chose not to
    fill), that asset is silently excluded from the mean on that date,
    which reweights the surviving assets to sum to 1 -- itself a
    defensible choice, in contrast to fabricating a price.
    """
    asset_returns = price_panel.pct_change()
    portfolio_simple = asset_returns.mean(axis=1, skipna=True).dropna()
    if return_type == "simple":
        return portfolio_simple
    elif return_type == "log":
        return np.log1p(portfolio_simple)
    else:
        raise ValueError(return_type)


def compute_stats(returns, rf=0.0, periods_per_year=252, return_type="simple"):
    """
    Annualised return, annualised volatility, Sharpe ratio and maximum
    drawdown from a return series.

    Parameters
    ----------
    returns : Series
        Portfolio returns, simple or log per `return_type`.
    rf : float or Series
        Per-period risk-free rate already aligned to `returns`' index. Used
        only for the Sharpe ratio's excess return, not for the headline
        annualised return. Pass 0.0 for the "Sharpe against zero" baseline.
    periods_per_year : float
        252 for the naive assumption; the panel's own observed trading-day
        count for the corrected version (Step 7).
    return_type : "simple" or "log"
        For simple returns the annualised return is the arithmetic mean
        scaled by periods_per_year -- this OVERSTATES the compounded growth
        rate because of volatility drag (Jensen's inequality on the concave
        log-wealth function). For log returns, the arithmetic mean scaled by
        periods_per_year already equals the annualised geometric growth
        rate, which is why switching return_type is this waterfall's
        built-in arithmetic-vs-geometric correction (Step 8).
    """
    returns = returns.dropna()
    if isinstance(rf, pd.Series):
        rf = rf.reindex(returns.index).fillna(0.0)
    excess = returns - rf

    if return_type == "simple":
        wealth = (1 + returns).cumprod()
    elif return_type == "log":
        wealth = np.exp(returns.cumsum())
    else:
        raise ValueError(return_type)

    ann_return = returns.mean() * periods_per_year
    ann_vol = excess.std(ddof=1) * np.sqrt(periods_per_year)
    ann_excess_return = excess.mean() * periods_per_year
    sharpe = ann_excess_return / ann_vol if ann_vol > 0 else np.nan

    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1

    return {
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": drawdown.min(),
    }


waterfall_rows = {}

def record_step(label, portfolio_returns, **kwargs):
    """Compute and stash one waterfall row; returns the stats dict too."""
    stats = compute_stats(portfolio_returns, **kwargs)
    waterfall_rows[label] = stats
    print(label, "->", {k: round(v, 4) for k, v in stats.items()})
    return stats


In [ ]:
def naive_ffill(price_panel):
    """Blindly forward-fill every gap, regardless of why it exists."""
    return price_panel.ffill()


def defensible_fill(price_panel, max_gap=1):
    """
    Forward-fill only short (<= max_gap row) gaps -- the signature of one
    market being closed while another traded (e.g. a Japanese public
    holiday that is not a US holiday, or vice versa), where "price
    unchanged, no trade occurred" is a defensible reading. A longer gap is
    NOT filled and is reported instead of fabricated; A1's loader already
    flagged anything that looked like a broken ticker, so a long gap
    surviving to here is treated as genuinely missing.
    """
    filled = price_panel.ffill(limit=max_gap)
    still_missing = filled.isna().sum()
    if still_missing.sum() > 0:
        print(f"Still missing after a {max_gap}-trading-day defensible fill "
              f"(left as NaN, not fabricated):")
        print(still_missing[still_missing > 0])
    else:
        print(f"No gaps longer than {max_gap} trading day(s) remained after the defensible fill.")
    return filled


In [ ]:
def flag_extreme_returns(asset_returns, z_thresh=6.0):
    """
    Flag single-asset daily returns more than `z_thresh` standard deviations
    from THAT ASSET's own mean -- a per-asset, not a blanket, threshold,
    since assets have very different typical volatility.

    Implemented without DataFrame.stack(), whose default NaN-dropping
    behaviour changed across pandas versions and silently returned every
    cell (including NaNs) rather than only the flagged ones in an earlier
    draft of this function -- verified with a synthetic bad print before
    trusting it on real data.
    """
    z = (asset_returns - asset_returns.mean()) / asset_returns.std(ddof=1)
    mask = z.abs() > z_thresh
    records = []
    for ticker in asset_returns.columns:
        hits = asset_returns.loc[mask[ticker].fillna(False), ticker]
        for date, value in hits.items():
            records.append({"date": date, "ticker": ticker, "return": value,
                             "z_score": z.loc[date, ticker]})
    flagged = pd.DataFrame(records, columns=["date", "ticker", "return", "z_score"])
    return flagged.sort_values("date").reset_index(drop=True)


# Market-wide (or sector-wide) episodes inside our 2016-2025 sample. A
# flagged observation that falls inside one of these windows is presumed a
# genuine move, not a data error, unless investigation of the surrounding
# prices says otherwise.
#
# This list covers globally-synchronised events likely to show up across a
# US-equity-plus-commodity-miner universe. It deliberately omits anything
# specific to the previous version of this notebook's fifth asset (Toyota,
# a Japan-listed automaker) -- e.g. its Aug 2024 BoJ/carry-trade shock has
# no obvious reason to move BHP.AX. If a genuinely flagged observation from
# your own run does not match anything here (for instance a BHP.AX-specific
# move tied to an iron-ore price shock, a China property-market scare, or a
# company announcement), ADD it here rather than silently letting the
# correction step forward-fill over a real, well-documented market move --
# exactly the mistake an earlier version of this notebook's build process
# caught and documents in Part C.
KNOWN_MARKET_EVENTS = [
    ("2018-02-02", "2018-02-09", "Volmageddon -- XIV unwind, VIX spike"),
    ("2018-12-01", "2018-12-26", "Q4 2018 growth-scare sell-off"),
    ("2020-02-20", "2020-04-07", "COVID-19 crash and initial rebound"),
    ("2020-09-01", "2020-09-08", "Sept 2020 tech pullback"),
    ("2020-11-09", "2020-11-09", "Pfizer/BioNTech vaccine efficacy announcement"),
    ("2022-01-01", "2022-10-31", "2022 rate-hike bear market"),
    ("2023-03-08", "2023-03-15", "SVB / regional bank stress"),
    ("2024-11-05", "2024-11-08", "2024 US presidential election result"),
]


def classify_flagged(flagged, events=KNOWN_MARKET_EVENTS):
    def event_for(date):
        for start, end, name in events:
            if pd.Timestamp(start) <= date <= pd.Timestamp(end):
                return name
        return None
    flagged = flagged.copy()
    flagged["known_event"] = flagged["date"].apply(event_for)
    return flagged


In [ ]:
def convert_aud_to_usd(aud_price_series, audusd_rate):
    """
    `audusd_rate` (Yahoo ticker "AUDUSD=X") quotes USD per 1 AUD -- the
    OPPOSITE convention from "JPY=X", which quotes JPY per 1 USD. AUD, GBP,
    EUR and NZD are quoted "USD per 1 unit of foreign currency" by
    historical ("commonwealth") convention, while JPY (and most other
    currencies) are quoted "foreign currency per 1 USD". Converting to USD
    therefore means MULTIPLYING an AUD price by this rate, not dividing by
    it the way the JPY case does -- reusing that division logic unchanged
    here would silently produce a price series wrong by a factor of roughly
    rate^2, not a rounding error. Caught by checking the conversion against
    a manually-verified single date before trusting it on the full series.
    Rate gaps are forward-filled since FX trades continuously and a missing
    print just means "unchanged since the last quote", unlike an equity halt.
    """
    return aud_price_series * audusd_rate.reindex(aud_price_series.index).ffill()


def align_risk_free(returns_index, rf_series, direction="backward"):
    """
    Align a sparsely-observed risk-free series (T-bill yields are not
    published on every trading day) onto `returns_index` via an as-of merge.

    direction="backward" (correct): use the most recent rate that was
    actually known as of each date -- no future information used.
    direction="forward" (the look-ahead bug): uses the NEXT rate the market
    had not yet seen. Included only to measure the bug's size before
    discarding it.

    Bug fixed here: an earlier version built the "date" column for each
    side differently -- `rename(columns={"index": "date"})` on one side
    assumed `reset_index()` always produces a column literally called
    "index", which is only true when the source index has no name. Our
    price/return series inherit the name "Date" from yfinance, so that
    assumption silently failed on real data (KeyError from merge_asof, not
    caught by the earlier synthetic test because that test used an
    already-unnamed DatetimeIndex). `rename_axis("date")` before
    `reset_index()` sidesteps the issue entirely: the resulting column is
    always called "date", whatever the source index was named or not.
    """
    left = pd.DataFrame(index=returns_index).rename_axis("date").reset_index()
    right = rf_series.sort_index().rename("rf").rename_axis("date").reset_index()
    aligned = pd.merge_asof(left, right, on="date", direction=direction).set_index("date")["rf"]
    return aligned


### Steps 0-1: naive baseline, then dividend/split-adjusted prices

In [ ]:
naive_prices = naive_ffill(raw_close)
r0 = equal_weight_portfolio_returns(naive_prices)
s0 = record_step("0. Naive baseline (raw prices, blind ffill, Sharpe vs 0, 252d, simple/arith.)", r0)

adj_prices_naive_fill = naive_ffill(adj_close)
r1 = equal_weight_portfolio_returns(adj_prices_naive_fill)
s1 = record_step("1. + dividend/split-adjusted prices", r1)


### Step 2: defensible treatment of non-trading days

BHP.AX (Australian Securities Exchange) and the four US tickers do not share a holiday
calendar -- Australia Day, ANZAC Day, the King's Birthday and other AU-only public
holidays fall on days the US market is open, and the reverse also happens (e.g. the ASX
trades as normal on US Thanksgiving) -- so the raw joined panel has gaps on US-only and
Australia-only holidays alike. Blindly forward-filling every gap (Step 0/1) treats a
several-day data outage the same as a single foreign holiday. We instead forward-fill
only single-day gaps and report anything longer rather than fabricate it.

In [ ]:
adj_prices_defensible = defensible_fill(adj_close, max_gap=1)
r2 = equal_weight_portfolio_returns(adj_prices_defensible)
s2 = record_step("2. + defensible non-trading-day treatment", r2)


### Step 3: investigate extreme returns

We flag any single-asset daily return more than 6 standard deviations from that asset's
own mean, then check each flagged date against known market-wide events in our sample.
A flag that falls inside a known event window is a genuine market move and is left alone
-- winsorising it would delete real information (per the brief's own warning). A flag
that does **not** correspond to a known event is a candidate data error; we correct those
by replacing the price level with the most recent valid price *before* it (a
forward-fill), not the return, and only for the specific (date, ticker) pairs identified.

We deliberately do not use a two-sided interpolation here, even though it would produce a
smoother-looking "corrected" price. Linear interpolation between the surrounding prices
uses the *next* valid observation as well as the prior one -- so the corrected value on
the flagged date would depend on information not yet available as of that date, which is
exactly the look-ahead bias this notebook hunts for elsewhere (e.g. Step 5's risk-free
alignment bug below). Forward-filling from the last known good price uses only
information available at the time, at the cost of a slightly less accurate point estimate
of the "true" price on the corrected date -- the same trade-off Step 2 already accepted
for ordinary non-trading-day gaps. In our own sample this path never actually fires (every
flagged observation below falls inside a known event window, so nothing gets corrected),
so the choice is currently vacuous in effect -- but it is the principled, look-ahead-free
choice for any future run where it does.

One diagnostic worth watching for: a genuine one-sided crash does not fully reverse the
next day, while an isolated bad print typically shows as a paired anomaly -- an extreme
move immediately followed by a roughly offsetting one, as the series snaps back to its
true level. That pattern, if present, is evidence for "data error" over "real move".

In [ ]:
asset_returns_2 = adj_prices_defensible.pct_change()
flagged = flag_extreme_returns(asset_returns_2, z_thresh=6.0)
flagged_classified = classify_flagged(flagged)
print(f"{len(flagged_classified)} extreme return(s) flagged (|z| > 6):")
display(flagged_classified)

genuinely_unexplained = flagged_classified[flagged_classified["known_event"].isna()]

if genuinely_unexplained.empty:
    print("\nEvery flagged extreme return falls inside a known market-wide event window "
          "(or none were flagged at all); none is treated as a data error. No price "
          "correction is applied at this step -- we demonstrate the check rather than "
          "assert its result. A vendor that served, say, a single-day zero-price glitch "
          "on an illiquid micro-cap would be exactly the case that WOULD move this row.")
    adj_prices_clean = adj_prices_defensible
else:
    print(f"\n{len(genuinely_unexplained)} flagged observation(s) do not correspond to a "
          "known market-wide event and are candidates for a genuine data error:")
    display(genuinely_unexplained)
    adj_prices_clean = adj_prices_defensible.copy()
    for _, row in genuinely_unexplained.iterrows():
        adj_prices_clean.loc[row["date"], row["ticker"]] = np.nan
    adj_prices_clean = adj_prices_clean.ffill()
    print("Replaced the flagged, unexplained observation(s) above with the most recent "
          "valid price (forward-fill, not a two-sided interpolation) -- using only "
          "information available as of the flagged date, to avoid introducing a "
          "look-ahead bias into the very correction meant to fix a data error.")

r3 = equal_weight_portfolio_returns(adj_prices_clean)
s3 = record_step("3. + investigated extreme returns", r3)


### Step 4: currency inconsistency

Four of our five assets are USD; BHP.AX is quoted in AUD. Computing its return directly
from the AUD price and averaging it in with USD returns silently assumes 1 AUD of price
change is worth the same as 1 USD of price change, and ignores the AUD/USD exchange-rate
return entirely -- which a USD-based investor actually bears. We convert BHP's price
series to USD using the AUD/USD rate before computing any returns.

**A quoting-convention trap, distinct from the one an earlier version of this notebook's
JPY example hit.** Yahoo Finance's `JPY=X` ticker quotes JPY *per 1 USD*, so converting
to USD means *dividing* the JPY price by the rate. AUD follows the opposite,
"commonwealth" quoting convention: `AUDUSD=X` quotes USD *per 1 AUD* -- how many US
dollars one Australian dollar buys. Converting BHP's AUD price to USD therefore means
*multiplying* by the rate, not dividing -- reusing the JPY conversion function's division
logic here unchanged would silently produce a currency-converted price series wrong by a
factor of roughly `rate^2`, not a rounding error. We deliberately did not reuse
`convert_jpy_to_usd()` unchanged; `convert_aud_to_usd()` below multiplies instead, and
its docstring states which Yahoo ticker convention it assumes so the difference is not
silently lost.

In [ ]:
fx_raw, fx_adj, fx_dl_log = load_price_panel(["AUDUSD=X"], PRICE_START, PRICE_END,
                                              cache_dir=DATA_DIR / "fx")
audusd = fx_raw["AUDUSD=X"]
print("AUDUSD download log:", fx_dl_log)

prices_usd = adj_prices_clean.copy()
prices_usd["BHP.AX"] = convert_aud_to_usd(adj_prices_clean["BHP.AX"], audusd)

r4 = equal_weight_portfolio_returns(prices_usd)
s4 = record_step("4. + currency conversion (BHP.AX AUD -> USD)", r4)

### Steps 5-6: risk-free rate -- zero vs actual, and a look-ahead bug in how it's aligned

So far every Sharpe ratio above was computed against a risk-free rate of exactly zero.
We now bring in the actual US 3-month T-bill yield (`^IRX`, quoted annualised, in percent)
and convert it to a per-period rate. T-bill yields are not published on every trading day,
so aligning them onto our daily return index requires an as-of merge -- and that join
direction is a classic place for a look-ahead bug to hide: `direction="forward"` pairs a
return with a rate the market had not yet observed. We measure that bug's size first
(Step 5), then fix it (Step 6, `direction="backward"`), isolating the look-ahead effect
from the "zero vs actual rate" effect.

In [ ]:
rf_raw, rf_adj, rf_dl_log = load_price_panel(["^IRX"], PRICE_START, PRICE_END,
                                              cache_dir=DATA_DIR / "rf")
irx = rf_raw["^IRX"]   # annualised T-bill yield, in percent
print("^IRX download log:", rf_dl_log)

rf_daily = (1 + irx / 100) ** (1 / 252) - 1

rf_forward_buggy = align_risk_free(r4.index, rf_daily, direction="forward")
rf_backward = align_risk_free(r4.index, rf_daily, direction="backward")
lookahead_gap = (rf_forward_buggy - rf_backward).abs()
print(f"\nLook-ahead alignment gap: mean={lookahead_gap.mean():.8f}, "
      f"max={lookahead_gap.max():.8f}, nonzero on "
      f"{int((lookahead_gap > 0).sum())} of {len(lookahead_gap)} dates")

s5 = record_step("5. + risk-free rate, buggy forward-looking alignment", r4, rf=rf_forward_buggy)
s6 = record_step("6. + risk-free rate, corrected backward-looking alignment", r4, rf=rf_backward)


### Isolating one comparison: Sharpe against zero vs. Sharpe against the actual cash rate

The waterfall above already contains this comparison implicitly (Step 4 used rf = 0; Steps 5-6 use the actual T-bill rate), but the brief asks for it explicitly. To isolate *only* this one dimension -- holding the return series (`r4`), the annualisation factor (252) and the alignment method (backward, correct) all fixed -- we recompute the same portfolio's Sharpe ratio twice, changing nothing but the risk-free assumption.

In [ ]:
sharpe_vs_zero = compute_stats(r4, rf=0.0, periods_per_year=252)["sharpe"]
sharpe_vs_actual = compute_stats(r4, rf=rf_backward, periods_per_year=252)["sharpe"]

rf_comparison = pd.DataFrame({
    "Risk-free assumption": ["Zero (rf = 0)", "Actual cash rate (^IRX, backward-aligned)"],
    "Sharpe ratio": [sharpe_vs_zero, sharpe_vs_actual],
}).set_index("Risk-free assumption")
rf_comparison["Change from zero-rf Sharpe"] = rf_comparison["Sharpe ratio"] - sharpe_vs_zero

display(rf_comparison.style.format({
    "Sharpe ratio": "{:.3f}", "Change from zero-rf Sharpe": "{:+.3f}",
}))

### Step 7: annualisation factor -- 252 assumed vs actual trading days observed

In [ ]:
actual_ppy = len(r4) / ((r4.index[-1] - r4.index[0]).days / 365.25)
print(f"Actual periods/year in this panel: {actual_ppy:.2f} (naive assumption was 252)")

rf_daily_actual = (1 + irx / 100) ** (1 / actual_ppy) - 1
rf_backward_actual = align_risk_free(r4.index, rf_daily_actual, direction="backward")

s7 = record_step("7. + actual trading-day annualisation factor", r4,
                  rf=rf_backward_actual, periods_per_year=actual_ppy)


### Step 8: return definition -- simple/arithmetic vs log/geometric

An arithmetic mean of simple returns, scaled to an annual figure, systematically
overstates the return an investor actually compounds to, because volatility drags the
geometric growth rate below the arithmetic mean (Jensen's inequality on the concave
log-wealth function). Switching to log returns is not a separate "extra" correction on
top of computing a geometric mean -- the arithmetic mean of log returns already **is**
the annualised geometric growth rate, which is why this step recomputes everything with
`return_type="log"` rather than adding a fifth statistic.

Note what this step does and does not change: the underlying portfolio -- and therefore
its wealth path and maximum drawdown -- is identical to Step 7's; only the *summary
statistic* used to describe its annualised return changes. (An earlier draft of
`equal_weight_portfolio_returns` computed the log-return version by averaging each
asset's own log return across the cross-section, which is a subtly different quantity
from this portfolio's own return on a log scale, by Jensen's inequality -- confirmed with
a synthetic test where that mistake alone moved measured drawdown, with nothing to do
with simple-vs-log at all. Fixed by taking `log1p` of the portfolio's own simple return
instead.)


In [ ]:
log_returns = equal_weight_portfolio_returns(prices_usd, return_type="log")
s8 = record_step("8. + log returns (arithmetic mean = geometric growth rate)", log_returns,
                  rf=rf_backward_actual, periods_per_year=actual_ppy, return_type="log")


### The waterfall table

One row per correction, cumulative -- the deliverable for this section.

In [ ]:
waterfall = pd.DataFrame(waterfall_rows).T
waterfall.columns = ["Ann. return", "Ann. vol", "Sharpe", "Max drawdown"]
display(waterfall.style.format({
    "Ann. return": "{:.2%}", "Ann. vol": "{:.2%}",
    "Sharpe": "{:.3f}", "Max drawdown": "{:.2%}",
}))


### Do the corrections offset each other?

Compare the naive baseline to the fully corrected row (the *net* change), then compare
that against the single largest step-to-step move for each statistic. If the net change is
noticeably smaller than the largest single step, some corrections pushed the numbers in
opposite directions -- meaning the naive-vs-final comparison alone understates how wrong
the naive figure actually was at any single point in the pipeline.

In [ ]:
net_change = waterfall.iloc[-1] - waterfall.iloc[0]
step_changes = waterfall.diff().iloc[1:]

print("Net change, naive baseline -> fully corrected:")
print(net_change.round(4))

print("\nLargest single-step move per statistic:")
for col in waterfall.columns:
    step_name = step_changes[col].abs().idxmax()
    largest = step_changes.loc[step_name, col]
    ratio = abs(net_change[col]) / abs(largest) if largest != 0 else np.nan
    flag = " <-- net change is SMALLER than this single step: corrections partly offset" \
        if abs(net_change[col]) < abs(largest) - 1e-12 else ""
    print(f"  {col}: largest move at '{step_name}' ({largest:+.4f}); "
          f"net/largest ratio = {ratio:.2f}{flag}")


### A2 discussion: which correction mattered most?

TODO once run: identify which single correction (dividend/split adjustment,
non-trading-day treatment, extreme-return investigation, AUD currency conversion,
risk-free alignment, annualisation factor, or return definition) moved each of the four
statistics (annualised return, annualised volatility, Sharpe ratio, maximum drawdown) the
most, quoting the before/after values from the waterfall table above.

Then answer the brief's second question directly: was the naive Sharpe ratio a modest
overstatement of the fully corrected figure, or a fundamentally different answer? State
the net change (naive row minus fully-corrected row) and compare it explicitly against
the largest single step from the offsetting-corrections check below -- if the net change
is smaller than the largest single step, some corrections pushed in opposite directions,
and you should say so explicitly rather than letting the small net change imply the naive
figure was basically fine.

### Survivorship bias

**What it is.** Every asset in our universe -- Apple, JPMorgan, ExxonMobil, Johnson &
Johnson, BHP -- is a company that not only still exists today but is a well-known,
currently-thriving large-cap. We selected them with the benefit of nine years of
hindsight. A retail data vendor like Yahoo Finance only serves prices for tickers that
are still listed (or were, under the same symbol, until a known and gracefully-handled
event such as a merger); it does not serve a point-in-time-correct universe of *everything
that was investable* on 1 January 2016, including firms that have since been delisted,
gone bankrupt, or been acquired at a distressed price. We cannot fix this with data
available to us -- `yfinance` simply has no route to a name that no longer trades.

**Direction of the bias.** Upward on return and Sharpe ratio, and understated on
volatility and drawdown. A portfolio that could only ever hold survivors never
experiences one of its constituents going to zero, so its realised return, Sharpe ratio
and worst drawdown are all more flattering than what an investor holding the *actual*,
point-in-time investable universe in 2016 would have experienced.

**Rough magnitude.** Lecture 2 cites a calibrated delisting simulation on a broad,
systematically-sampled S&P 500 backtest putting survivorship bias at roughly +2.7% p.a. of
phantom return. Our case is arguably worse on this dimension, not better: we did not
sample systematically at all -- we hand-picked five globally recognised, multi-decade
survivors specifically because they were recognisable, which is a much stronger
hindsight-driven selection filter than "was still in the S&P 500 index at the download
date". We would expect our true survivorship effect, if it could be measured, to sit at or
above that +2.7% p.a. anchor rather than below it, though we have no way to quantify our
own figure without a point-in-time delisted-security database, which is not available
through a free retail vendor.

## A3 — Baseline risk report

Using the corrected data throughout: `prices_usd` from A2 (adjusted for corporate actions,
defensibly filled, investigated for data errors, and currency-converted) is our clean
price panel from here on, and `r4` (the equal-weight portfolio return computed from it in
A2 Step 4) is our portfolio return series. We add a per-asset return panel alongside it.

In [ ]:
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns

asset_returns_final = prices_usd.pct_change()
portfolio_returns = r4   # from A2 Step 4: fully corrected, currency-converted

returns_for_a3 = {t: asset_returns_final[t] for t in TICKERS}
returns_for_a3["Portfolio (1/N)"] = portfolio_returns

print("Series available for A3:", list(returns_for_a3.keys()))


### Return characteristics: moments and normality

Mean, volatility, skewness and excess kurtosis for each asset and the portfolio, plus a
Jarque-Bera test of normality (a test built directly from sample skewness and kurtosis,
so its result should read consistently with the moments next to it).

In [ ]:
def compute_return_moments(returns_dict, periods_per_year=252):
    """
    Mean, volatility (daily and annualised), skewness, excess kurtosis and a
    Jarque-Bera normality test for each return series in `returns_dict`.
    """
    rows = {}
    for name, r in returns_dict.items():
        r = pd.Series(r).dropna()
        jb_stat, jb_p = stats.jarque_bera(r)
        rows[name] = {
            "mean_daily": r.mean(),
            "vol_daily": r.std(ddof=1),
            "ann_return": r.mean() * periods_per_year,
            "ann_vol": r.std(ddof=1) * np.sqrt(periods_per_year),
            "skewness": stats.skew(r),
            "excess_kurtosis": stats.kurtosis(r),   # fisher=True by default: 0 = normal
            "jarque_bera_stat": jb_stat,
            "jarque_bera_pvalue": jb_p,
        }
    return pd.DataFrame(rows).T


moments_table = compute_return_moments(returns_for_a3, periods_per_year=actual_ppy)
display(moments_table.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))

n_reject_normality = (moments_table["jarque_bera_pvalue"] < 0.05).sum()
print(f"\n{n_reject_normality} of {len(moments_table)} series reject normality at the 5% level.")


TODO once run: report each asset's and the portfolio's Jarque-Bera p-value, skewness
and excess kurtosis from the table above, and state whether normality is rejected at the
5% level (fat tails in daily equity/commodity returns make this the expected outcome).
Comment on which asset shows the most extreme kurtosis and why that might be -- an asset
with concentrated commodity/China exposure like BHP.AX could plausibly show fatter tails
around iron-ore-price or China-macro shocks specifically, distinct from the US assets'
own idiosyncratic drivers. Then explain what the fat-tailed, non-normal result implies
for any risk measure (like the parametric VaR/ES below) that assumes normality, and
comment on whether the portfolio's skewness differs from its constituents' skewness
individually -- if it does, that is usually evidence the assets crash together during a
systemic shock even though they look more independent in ordinary times.

### Value at Risk and Expected Shortfall

Two methods, at 95% and 99% confidence:

- **Historical (empirical):** the actual sample quantile of the loss distribution, and the
  average loss beyond it. Makes no distributional assumption, but is only as good as the
  historical sample's coverage of tail events.
- **Parametric (Gaussian):** assumes returns are normally distributed and computes VaR/ES
  from the sample mean and standard deviation in closed form. Cheap and smooth, but
  exactly the assumption the moments table above is testing.

Both are reported as positive loss numbers (VaR95 = 5.2% means "a 5.2% loss is exceeded
5% of the time").

In [ ]:
def historical_var_es(returns, alpha=0.95):
    """Empirical VaR/ES as positive loss numbers at confidence level alpha."""
    r = pd.Series(returns).dropna()
    q = r.quantile(1 - alpha)
    var = -q
    es = -r[r <= q].mean()
    return var, es


def parametric_var_es(returns, alpha=0.95):
    """Gaussian (parametric) VaR/ES as positive loss numbers, from sample mean/vol."""
    r = pd.Series(returns).dropna()
    mu, sigma = r.mean(), r.std(ddof=1)
    q = norm.ppf(1 - alpha)
    var = -(mu + sigma * q)
    es = -mu + sigma * norm.pdf(q) / (1 - alpha)
    return var, es


def var_es_table(returns_dict, confidence_levels=(0.95, 0.99)):
    rows = []
    for name, r in returns_dict.items():
        r = pd.Series(r).dropna()
        for alpha in confidence_levels:
            hv, he = historical_var_es(r, alpha)
            pv, pe = parametric_var_es(r, alpha)
            rows.append({"asset": name, "confidence": f"{alpha:.0%}",
                         "method": "historical", "VaR": hv, "ES": he})
            rows.append({"asset": name, "confidence": f"{alpha:.0%}",
                         "method": "parametric", "VaR": pv, "ES": pe})
    return pd.DataFrame(rows)


var_es = var_es_table(returns_for_a3, confidence_levels=(0.95, 0.99))
var_es_wide = var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                  values=["VaR", "ES"])
display(var_es_wide.style.format("{:.2%}"))

# Where the two methods disagree, and by how much
disagreement = (var_es.pivot_table(index=["asset", "confidence"], columns="method", values="ES")
                .assign(gap=lambda d: d["historical"] - d["parametric"]))
print("\nHistorical minus parametric ES (positive = parametric understates the tail):")
display(disagreement.style.format("{:.2%}"))


TODO once run: report the historical-minus-parametric ES gap for each asset and the
portfolio at 95% and 99% from the table above, and state whether the gap is positive
everywhere (i.e. the Gaussian assumption understates tail risk uniformly across this
universe). Identify which single asset shows the largest gap in absolute terms, and
whether the gap grows moving from 95% to 99% (it should, if the fat-tailedness found in
the moments table is real). State plainly which method -- historical or parametric --
you would report to a risk committee and why, tying the answer back to the Jarque-Bera
result above rather than treating the two cells as independent.

### Drawdown analysis

In [ ]:
def drawdown_series(returns):
    """
    Full drawdown path, plus the maximum drawdown and its peak/trough dates.
    Peak is the last date at or before the trough where wealth equalled its
    running maximum (i.e. where the subsequent decline actually started).
    """
    wealth = (1 + pd.Series(returns).dropna()).cumprod()
    running_max = wealth.cummax()
    dd = wealth / running_max - 1
    trough_date = dd.idxmin()
    max_dd = dd.loc[trough_date]
    peak_date = wealth.loc[:trough_date].idxmax()
    return dd, {"max_drawdown": max_dd, "peak_date": peak_date, "trough_date": trough_date}


portfolio_dd, portfolio_dd_info = drawdown_series(portfolio_returns)
print("Portfolio maximum drawdown:")
for k, v in portfolio_dd_info.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(portfolio_dd.index, portfolio_dd.values * 100, 0, color="#C0392B", alpha=0.5)
ax.plot(portfolio_dd.index, portfolio_dd.values * 100, color="#C0392B", linewidth=0.8)
ax.axvline(portfolio_dd_info["peak_date"], color="black", linestyle="--", linewidth=1,
           label=f"peak ({portfolio_dd_info['peak_date'].date()})")
ax.axvline(portfolio_dd_info["trough_date"], color="black", linestyle=":", linewidth=1,
           label=f"trough ({portfolio_dd_info['trough_date'].date()})")
ax.set_title("Portfolio drawdown, 1/N equal-weight, corrected data")
ax.set_xlabel("date")
ax.set_ylabel("drawdown (%)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()


TODO once run: report the actual peak date, trough date and maximum drawdown
percentage from the code above, and identify which entry in `KNOWN_MARKET_EVENTS` (A2) it
falls inside. Comment on whether the timing matches the well-documented external date for
that event (e.g. if it is the COVID crash, 23 March 2020 is the widely-reported historical
bottom of the broad US equity market) -- this is a useful external correctness check on
the drawdown code itself, independent of anything internal to this notebook. Note any
other visible drawdown episodes in the chart and whether they land where the event list
predicts.

### Correlation structure

Full-sample correlation matrix, then a rolling view -- diversification is a claim about
the *whole* sample, but the number that actually matters to a risk manager is whether it
holds up exactly when it is needed, during the worst drawdown.

In [ ]:
corr_matrix = asset_returns_final[TICKERS].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            square=True, ax=ax, cbar_kws={"label": "correlation"})
ax.set_title("Full-sample correlation matrix (corrected USD returns)")
plt.tight_layout()
plt.show()

display(corr_matrix.style.format("{:.3f}"))


In [ ]:
def rolling_avg_pairwise_corr(returns_df, window=60):
    """
    Average pairwise correlation across every asset pair, in a rolling
    window -- a single diversification-strength indicator over time.
    """
    cols = list(returns_df.columns)
    pairs = [(a, b) for i, a in enumerate(cols) for b in cols[i + 1:]]
    rolling_corrs = pd.DataFrame({
        f"{a}-{b}": returns_df[a].rolling(window).corr(returns_df[b])
        for a, b in pairs
    })
    avg_corr = rolling_corrs.mean(axis=1)
    return avg_corr, rolling_corrs


ROLLING_CORR_WINDOW = 60
avg_corr, pairwise_corrs = rolling_avg_pairwise_corr(asset_returns_final[TICKERS],
                                                      window=ROLLING_CORR_WINDOW)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                          gridspec_kw={"height_ratios": [1, 1.3]})

axes[0].fill_between(portfolio_dd.index, portfolio_dd.values * 100, 0,
                      color="#C0392B", alpha=0.4)
axes[0].set_ylabel("drawdown (%)")
axes[0].set_title("Portfolio drawdown vs. rolling average pairwise correlation")

axes[1].plot(avg_corr.index, avg_corr.values, color="#123F69", linewidth=1.2)
axes[1].axhline(avg_corr.mean(), color="grey", linestyle=":", linewidth=1,
                 label=f"full-sample average ({avg_corr.mean():.2f})")
axes[1].set_ylabel(f"{ROLLING_CORR_WINDOW}d avg pairwise correlation")
axes[1].set_xlabel("date")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.axvspan(portfolio_dd_info["peak_date"], portfolio_dd_info["trough_date"],
               color="black", alpha=0.08, label="max-drawdown window")

plt.tight_layout()
plt.show()


In [ ]:
dd_window_corr = avg_corr.loc[portfolio_dd_info["peak_date"]:portfolio_dd_info["trough_date"]].mean()
full_sample_corr = avg_corr.mean()
print(f"Average pairwise correlation during the max-drawdown window "
      f"({portfolio_dd_info['peak_date'].date()} to {portfolio_dd_info['trough_date'].date()}): "
      f"{dd_window_corr:.3f}")
print(f"Average pairwise correlation over the full sample: {full_sample_corr:.3f}")
print(f"Difference: {dd_window_corr - full_sample_corr:+.3f} "
      f"({'higher' if dd_window_corr > full_sample_corr else 'lower'} during the drawdown)")


TODO once run: report the full-sample average pairwise correlation and the average
pairwise correlation during the maximum-drawdown window from the code above, and state
whether correlation rose during the crisis (the "diversification fails when it's needed
most" pattern Lecture 6 predicts) or not. Comment specifically on BHP.AX's full-sample
correlation with the four US assets -- given BHP's commodity/China exposure is
economically quite different from the four US large-caps, check whether it sits at the
low end of the correlation matrix (evidence of genuine diversification value in normal
times) and, separately, whether that low correlation survives the COVID window or spikes
along with everything else. Do not assume the answer has to match whatever the original
five-asset (Toyota) version of this notebook found -- BHP's risk drivers are different,
and this needs checking on its own data.

## A4 — The equal-weight benchmark

**Rebalancing rule: monthly.** On the first trading day of every calendar month, weights
are reset to exactly 1/N; between rebalance dates, weights are left to drift with asset
prices (buy-and-hold). Monthly is a middle ground: annual lets drift accumulate for a
long time (by A3's evidence, correlation and volatility regimes can shift well within a
year), while daily rebalancing -- which is what A2 and A3 used implicitly throughout --
overstates what a real investor would do, since it assumes costless, frictionless trading
every single day. Monthly is also the more common real-world convention for a passive
benchmark.

Note that this makes A2/A3's portfolio (`r4`) and A4's benchmark genuinely different
series, not the same thing under a new name: A2/A3 used the daily-implicit version
throughout because A2's question was about data corrections, not rebalancing policy.
From here on, `benchmark_returns` (monthly-rebalanced) is what Part B is compared
against, per the brief's instruction.

In [ ]:
def rebalanced_portfolio_returns(price_panel, rebalance_freq="M", weights=None):
    """
    Simulate a portfolio rebalanced to target weights (default: equal
    weight, 1/N) on the first trading day of every `rebalance_freq`
    period (a pandas period alias -- "M" monthly, "Q" quarterly, "A"
    annual), drifting freely with asset prices between rebalance dates.

    Verified against a synthetic panel before use: rebalancing on every
    single day exactly reproduces `equal_weight_portfolio_returns()` (max
    abs difference ~3e-18, floating-point noise), and a one-rebalance
    single-month scenario matches manual buy-and-hold arithmetic exactly.

    Returns
    -------
    portfolio_returns : Series, daily simple returns
    turnover : Series, one-way turnover on each date (0 except on
        rebalance dates, and 0 on the very first date too -- that is
        portfolio *initiation* from cash, not a rebalancing trade away
        from an existing position).
    """
    returns = price_panel.pct_change().dropna(how="all")
    tickers = list(returns.columns)
    n = len(tickers)
    if weights is None:
        weights = {t: 1.0 / n for t in tickers}
    target = np.array([weights[t] for t in tickers])

    period_arr = np.asarray(returns.index.to_period(rebalance_freq))
    is_rebalance_day = np.empty(len(period_arr), dtype=bool)
    is_rebalance_day[0] = True
    is_rebalance_day[1:] = period_arr[1:] != period_arr[:-1]

    R = returns[tickers].values
    w = target.copy()
    port_returns = np.empty(len(returns))
    turnover = np.empty(len(returns))

    for t in range(len(returns)):
        if is_rebalance_day[t]:
            turnover[t] = np.abs(target - w).sum() / 2.0
            w = target.copy()
        else:
            turnover[t] = 0.0
        r_t = np.nan_to_num(R[t], nan=0.0)
        port_ret = float(np.dot(w, r_t))
        port_returns[t] = port_ret
        w = w * (1 + r_t) / (1 + port_ret)

    return (pd.Series(port_returns, index=returns.index, name="benchmark"),
            pd.Series(turnover, index=returns.index, name="turnover"))


benchmark_returns, benchmark_turnover = rebalanced_portfolio_returns(prices_usd, rebalance_freq="M")
n_rebalances = (benchmark_turnover > 0).sum()
print(f"Monthly-rebalanced benchmark built: {len(benchmark_returns)} daily observations, "
      f"{n_rebalances} rebalance events.")
print(f"Average one-way turnover per rebalance event: {benchmark_turnover[benchmark_turnover > 0].mean():.2%}")


### Benchmark performance and risk profile

Same statistics as A3, reusing the same functions -- return moments and normality,
VaR/ES by two methods, and the drawdown series -- applied to the monthly-rebalanced
benchmark. We also place the daily-implicit portfolio (`r4`) from A2/A3 alongside it, so
the effect of the rebalancing policy itself is directly visible rather than asserted.

In [ ]:
benchmark_moments = compute_return_moments(
    {"Benchmark (1/N, monthly)": benchmark_returns, "Daily-implicit (A2/A3)": portfolio_returns},
    periods_per_year=actual_ppy,
)
display(benchmark_moments.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))


In [ ]:
benchmark_var_es = var_es_table(
    {"Benchmark (1/N, monthly)": benchmark_returns, "Daily-implicit (A2/A3)": portfolio_returns},
    confidence_levels=(0.95, 0.99),
)
benchmark_var_es_wide = benchmark_var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                                      values=["VaR", "ES"])
display(benchmark_var_es_wide.style.format("{:.2%}"))


In [ ]:
benchmark_dd, benchmark_dd_info = drawdown_series(benchmark_returns)
print("Benchmark (1/N, monthly) maximum drawdown:")
for k, v in benchmark_dd_info.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(portfolio_dd.index, portfolio_dd.values * 100, color="#9AA5B1", linewidth=0.9,
        label="daily-implicit (A2/A3)")
ax.plot(benchmark_dd.index, benchmark_dd.values * 100, color="#123F69", linewidth=1.1,
        label="benchmark (1/N, monthly-rebalanced)")
ax.set_title("Drawdown: monthly-rebalanced benchmark vs. daily-implicit portfolio")
ax.set_xlabel("date")
ax.set_ylabel("drawdown (%)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()


TODO once run: compare the monthly-rebalanced benchmark's annualised return,
volatility, VaR/ES and maximum drawdown against the daily-implicit A2/A3 portfolio's
figures side by side, and state how close they are. Report the actual number of
rebalance events and average one-way turnover from the "benchmark built" cell above, and
use the real figure to make the cost-of-turnover argument for monthly over daily
rebalancing explicitly, rather than asserting it in general terms.

### Frictions ignored

We are not required to model these, but we are required to know they are missing --
and to have a sense of their size rather than just naming them.

- **Transaction costs.** The benchmark trades on every rebalance date, at the average
  one-way turnover per event reported by the "benchmark built" cell above (roughly a few
  percent per month, driven by how far our five assets drift apart between rebalances).
  At even a conservative 5-10 bp round-trip cost on large-cap, liquid names, that is a
  small but nonzero annual drag -- and would be far larger for a less liquid universe or
  a more frequent rebalancing rule.
- **Bid-ask spread.** Every trade crosses the spread, which is not the same as a
  commission and is not visible in any exchange-reported closing price -- it is a real
  cost this notebook has no way to measure from daily OHLCV data alone.
- **Cash drag from imperfect rebalancing.** A real account cannot buy fractional shares
  in the exact proportions 1/N implies, and cannot rebalance at the exact instant the
  code specifies (the next tradable price is not the theoretical rebalance price). Both
  push the achievable return slightly below what this simulation reports.

None of these favour the benchmark disproportionately -- if anything, an actively
rebalanced or optimised Part B strategy typically trades *more* than 1/N, so these same
frictions bite it harder. That asymmetry is itself relevant context for evaluating Part B
against this benchmark, not just a disclaimer.

### Adopting the benchmark for Part B

`benchmark_returns` (1/N, monthly-rebalanced) is the reference series for every Part B
result that can sensibly be compared against it, per the brief. A small reusable
comparison function, built once here rather than re-derived per extension.

In [ ]:
def benchmark_comparison_table(strategy_returns_dict, benchmark=benchmark_returns,
                               periods_per_year=None, confidence_levels=(0.95, 0.99)):
    """
    Side-by-side moments + VaR/ES + drawdown for one or more Part B
    strategies against the 1/N benchmark, using the exact same functions
    as A3/A4 so every comparison in this notebook is computed identically.
    """
    ppy = periods_per_year if periods_per_year is not None else actual_ppy
    series = {"Benchmark (1/N, monthly)": benchmark, **strategy_returns_dict}

    moments = compute_return_moments(series, periods_per_year=ppy)
    var_es = var_es_table(series, confidence_levels=confidence_levels)
    var_es_wide = var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                     values=["VaR", "ES"])

    dd_rows = {}
    for name, r in series.items():
        _, info = drawdown_series(r)
        dd_rows[name] = info
    drawdowns = pd.DataFrame(dd_rows).T

    return moments, var_es_wide, drawdowns


# Smoke test: calling it with no extra strategies should just reproduce the
# benchmark's own row from the tables above.
_moments_check, _var_es_check, _dd_check = benchmark_comparison_table({})
display(_moments_check.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))


## Extension 1 -- Portfolio optimisation (Category B, option B1)

### Methodological choices

**Short selling: not permitted (long-only).** Two reasons. First, Jagannathan and Ma
(2003) show a no-short constraint is mathematically equivalent to shrinking the
covariance matrix -- it is not merely a practical restriction, it is itself a defence
against estimation error, and we have already established (A3) that this universe has
heavy fat tails and a correlation structure that is anything but stable through time.
Second, a long-only mandate is the realistic default for the kind of investor this
benchmark exercise represents.

**Expected returns: the simple historical sample mean, annualised.** This is
deliberately the weakest possible choice, and we are explicit about that rather than
hiding it: Chopra and Ziemba (1993) find errors in mu are roughly an order of magnitude
more damaging to portfolio choice than errors in the covariance matrix, and Lecture 2 has
already established that returns are close to unforecastable at this horizon. We use it
anyway, precisely so the in-sample/out-of-sample comparison below can show what that
weak link actually costs, rather than assuming a better forecast we do not have.

**Covariance estimator: Ledoit-Wolf shrinkage**, not the raw sample covariance. Even
though T >> N here (roughly 2,000+ daily observations against 5 assets, unlike the
high-dimensional factor-zoo setting shrinkage is usually motivated by), shrinkage is
close to costless and never hurts conditioning -- it is the honest baseline any more
elaborate covariance method should be measured against, per Lecture 3.

In [ ]:
from sklearn.covariance import LedoitWolf

def estimate_covariance(returns_df, method="ledoit_wolf", periods_per_year=252):
    """
    Annualised covariance matrix from a daily returns panel.

    method : "sample" (raw sample covariance, shown only for comparison)
             or "ledoit_wolf" (shrinkage estimator, our default -- see
             justification above).
    """
    clean = returns_df.dropna()
    X = clean.values
    if method == "sample":
        cov = np.cov(X, rowvar=False, ddof=1)
    elif method == "ledoit_wolf":
        cov = LedoitWolf().fit(X).covariance_
    else:
        raise ValueError(method)
    return pd.DataFrame(cov, index=clean.columns, columns=clean.columns) * periods_per_year


In [ ]:
from scipy.optimize import minimize

def optimize_portfolio(mu, cov, objective="min_variance", rf=0.0, target_return=None,
                       long_only=True):
    """
    Solve for portfolio weights under one of three objectives.

    objective : "min_variance", "max_sharpe", or "risk_parity"
    target_return : if given (min_variance only), adds an equality constraint
        w @ mu == target_return -- used to trace the efficient frontier.
    long_only : constrains 0 <= w_i <= 1 for min_variance/max_sharpe (see
        justification above). Not applicable to risk_parity, which requires
        strictly positive weights by construction (the log-barrier term).

    Risk parity uses the Spinu (2013) convex reformulation from Lecture 6:
    minimise 0.5 w'Sigma w - sum(b_i log w_i), which recovers equal risk
    contributions after rescaling to sum to 1. Bounds must be (0, None),
    NOT (0, 1): an earlier version bounded w_i <= 1, which silently clips
    every weight to exactly 1.0 whenever the true unconstrained scale
    exceeds 1 (the problem is not scale-invariant before normalising) --
    all weights end up equal by coincidence of hitting the same clipped
    bound, which looks plausible but is not actually risk parity. Caught
    by checking realised risk contributions directly against a synthetic
    portfolio with known unequal volatilities, rather than trusting solver
    convergence alone.

    Returns
    -------
    weights : Series indexed like `mu`
    result : the scipy OptimizeResult, for convergence diagnostics
    """
    tickers = mu.index if isinstance(mu, pd.Series) else None
    mu_v, cov_v = np.asarray(mu), np.asarray(cov)
    n = len(mu_v)
    x0 = np.full(n, 1.0 / n)

    if objective == "risk_parity":
        b = np.full(n, 1.0 / n)
        bounds = [(1e-8, None)] * n
        def rp_obj(w):
            return 0.5 * w @ cov_v @ w - np.sum(b * np.log(w))
        res = minimize(rp_obj, x0, method="SLSQP", bounds=bounds,
                       options={"maxiter": 1000, "ftol": 1e-14})
        w = res.x / res.x.sum()
        return (pd.Series(w, index=tickers) if tickers is not None else w), res

    bounds = [(0.0, 1.0)] * n if long_only else [(-1.0, 1.0)] * n
    constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]
    if target_return is not None:
        constraints.append({"type": "eq", "fun": lambda w: w @ mu_v - target_return})

    if objective == "min_variance":
        obj = lambda w: w @ cov_v @ w
    elif objective == "max_sharpe":
        def obj(w):
            vol = np.sqrt(w @ cov_v @ w)
            return -(w @ mu_v - rf) / vol if vol > 1e-12 else 1e6
    else:
        raise ValueError(objective)

    res = minimize(obj, x0, method="SLSQP", bounds=bounds, constraints=constraints,
                   options={"maxiter": 1000, "ftol": 1e-12})
    if not res.success:
        print(f"WARNING: optimizer did not converge cleanly for {objective}: {res.message}")
    w = res.x
    return (pd.Series(w, index=tickers) if tickers is not None else w), res


def risk_contributions(w, cov):
    """Euler risk contributions: sums exactly to portfolio volatility."""
    w_v, cov_v = np.asarray(w), np.asarray(cov)
    port_vol = np.sqrt(w_v @ cov_v @ w_v)
    marginal = (cov_v @ w_v) / port_vol
    return pd.Series(w_v * marginal, index=w.index if isinstance(w, pd.Series) else None)


In [ ]:
mu_full = asset_returns_final[TICKERS].mean() * actual_ppy
cov_full = estimate_covariance(asset_returns_final[TICKERS], "ledoit_wolf", periods_per_year=actual_ppy)

w_minvar, res_minvar = optimize_portfolio(mu_full, cov_full, objective="min_variance")
w_maxsharpe, res_maxsharpe = optimize_portfolio(mu_full, cov_full, objective="max_sharpe")
w_riskparity, res_riskparity = optimize_portfolio(mu_full, cov_full, objective="risk_parity")
w_equal = pd.Series(1.0 / len(TICKERS), index=TICKERS)

for name, res in [("min_variance", res_minvar), ("max_sharpe", res_maxsharpe),
                  ("risk_parity", res_riskparity)]:
    print(f"{name}: converged = {res.success}")

weights_table = pd.DataFrame({
    "Min variance": w_minvar, "Max Sharpe": w_maxsharpe,
    "Risk parity": w_riskparity, "1/N": w_equal,
})
display(weights_table.style.format("{:.2%}"))

rc_table = pd.DataFrame({
    "Min variance": risk_contributions(w_minvar, cov_full) / np.sqrt(w_minvar.values @ cov_full.values @ w_minvar.values),
    "Max Sharpe": risk_contributions(w_maxsharpe, cov_full) / np.sqrt(w_maxsharpe.values @ cov_full.values @ w_maxsharpe.values),
    "Risk parity": risk_contributions(w_riskparity, cov_full) / np.sqrt(w_riskparity.values @ cov_full.values @ w_riskparity.values),
    "1/N": risk_contributions(w_equal, cov_full) / np.sqrt(w_equal.values @ cov_full.values @ w_equal.values),
})
print("\nRisk contribution shares (should be exactly equal within 'Risk parity' column):")
display(rc_table.style.format("{:.2%}"))


### Efficient frontier

In [ ]:
def efficient_frontier(mu, cov, n_points=40, long_only=True):
    """Sweep target returns, solving min-variance at each, to trace the frontier."""
    mu_v, cov_v = np.asarray(mu), np.asarray(cov)
    targets = np.linspace(mu_v.min(), mu_v.max(), n_points)
    rows = []
    for tr in targets:
        w, res = optimize_portfolio(mu, cov, objective="min_variance",
                                    target_return=tr, long_only=long_only)
        if res.success:
            w_v = np.asarray(w)
            rows.append({"target_return": tr, "return": w_v @ mu_v,
                        "vol": np.sqrt(w_v @ cov_v @ w_v)})
    return pd.DataFrame(rows)


frontier = efficient_frontier(mu_full, cov_full, n_points=40)
# Plot only the efficient (upper) half: at/above the global minimum-variance return
gmv_return = frontier.loc[frontier["vol"].idxmin(), "return"]
frontier_efficient = frontier[frontier["return"] >= gmv_return - 1e-9]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(frontier["vol"] * 100, frontier["return"] * 100, color="#9AA5B1",
        linewidth=1.2, linestyle="--", label="frontier (full, incl. inefficient half)")
ax.plot(frontier_efficient["vol"] * 100, frontier_efficient["return"] * 100,
        color="#123F69", linewidth=2, label="efficient frontier")

markers = {
    "Min variance": (w_minvar, "o", "#1B7F79"),
    "Max Sharpe": (w_maxsharpe, "^", "#C0392B"),
    "Risk parity": (w_riskparity, "s", "#8E6C1F"),
    "1/N benchmark": (w_equal, "D", "#E8871A"),
}
for label, (w, marker, color) in markers.items():
    w_v = w.values
    vol_pt = np.sqrt(w_v @ cov_full.values @ w_v) * 100
    ret_pt = (w_v @ mu_full.values) * 100
    ax.scatter([vol_pt], [ret_pt], marker=marker, s=110, color=color,
              edgecolor="black", linewidth=0.8, zorder=5, label=label)

ax.set_xlabel("annualised volatility (%)")
ax.set_ylabel("annualised return (%)")
ax.set_title("Efficient frontier (long-only, Ledoit-Wolf covariance, full sample)")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()


TODO once run: report the actual weights table and risk-contribution table above.
Identify which asset each of the four portfolios (min variance, max Sharpe, risk parity,
1/N) concentrates in and why that makes sense given each asset's own volatility and
expected-return ranking from A3 -- e.g. does min variance lean on whichever asset had the
lowest full-sample volatility, and does max Sharpe lean on whichever asset had the
highest full-sample mean return (and is that concentration something you'd actually want
to hold, given Chopra-Ziemba's warning about how noisy that input is)? Confirm risk
parity's risk contributions are all equal (~20% each) by construction, and quantify the
weight/risk-share gap for the 1/N benchmark (which assets carry more risk share than
capital share -- check whether BHP.AX, despite a modest capital weight, contributes a
disproportionate risk share given commodity-linked assets are often more volatile than
large-cap US equities). Comment on whether min variance's weights and risk contributions
are numerically identical (they should be, by the first-order condition at the
minimum-variance optimum) and explain why that is not a coincidence.

### In-sample vs. out-of-sample: the real subject of this option

We split the sample chronologically at 70/30 -- estimate mu and Sigma on the first 70% of
the data only, solve all three portfolios there, then hold those weights *fixed* and
apply them to the remaining 30%, never re-estimated. This is the walk-forward discipline
Lectures 2 and 3 insist on: an in-sample number is what the optimiser promised itself;
the out-of-sample number is what an investor actually would have earned. The 1/N
benchmark needs no such split -- it estimates nothing, so there is nothing for it to
overfit -- but we still slice `benchmark_returns` to the identical out-of-sample window
for a fair comparison.

In [ ]:
SPLIT_FRACTION = 0.70
returns_clean = asset_returns_final[TICKERS].dropna()
split_idx = int(len(returns_clean) * SPLIT_FRACTION)
in_sample_dates = returns_clean.index[:split_idx]
out_sample_dates = returns_clean.index[split_idx:]

print(f"In-sample:     {in_sample_dates[0].date()} to {in_sample_dates[-1].date()} "
      f"({len(in_sample_dates)} obs)")
print(f"Out-of-sample: {out_sample_dates[0].date()} to {out_sample_dates[-1].date()} "
      f"({len(out_sample_dates)} obs)")

in_sample_returns = returns_clean.loc[in_sample_dates]
out_sample_returns = returns_clean.loc[out_sample_dates]

mu_is = in_sample_returns.mean() * actual_ppy
cov_is = estimate_covariance(in_sample_returns, "ledoit_wolf", periods_per_year=actual_ppy)
# Average annualised risk-free rate actually prevailing in-sample -- passed into the
# max-Sharpe solve itself, not left at the default rf=0, which A2 already established
# is the naive mistake (Sharpe "against zero" vs "against the actual cash rate").
rf_annual_is = rf_backward_actual.reindex(in_sample_dates).mean() * actual_ppy
print(f"Average in-sample annualised risk-free rate: {rf_annual_is:.2%}")

oos_strategies = {}
is_claimed_sharpe = {}
for name, objective in [("Min variance", "min_variance"), ("Max Sharpe", "max_sharpe"),
                        ("Risk parity", "risk_parity")]:
    w, res = optimize_portfolio(mu_is, cov_is, objective=objective, rf=rf_annual_is)
    w_v = w.values
    vol_is = np.sqrt(w_v @ cov_is.values @ w_v)
    is_claimed_sharpe[name] = (w_v @ mu_is.values - rf_annual_is) / vol_is
    oos_strategies[name] = pd.Series(out_sample_returns.values @ w_v, index=out_sample_dates,
                                     name=name)

oos_benchmark = benchmark_returns.reindex(out_sample_dates).dropna()
oos_strategies["1/N benchmark"] = oos_benchmark

# Reuse compute_stats (A2) so the out-of-sample Sharpe is computed against the same
# properly-aligned actual risk-free rate as everywhere else in this notebook, not
# against an implicit zero.
oos_rows = {name: compute_stats(r, rf=rf_backward_actual, periods_per_year=actual_ppy)
            for name, r in oos_strategies.items()}
oos_stats = pd.DataFrame(oos_rows).T
oos_stats["in_sample_claimed_sharpe"] = pd.Series(is_claimed_sharpe)

display(oos_stats[["ann_return", "ann_vol", "sharpe", "max_drawdown",
                   "in_sample_claimed_sharpe"]].rename(columns={
    "ann_return": "OOS ann. return", "ann_vol": "OOS ann. vol",
    "sharpe": "OOS Sharpe", "max_drawdown": "OOS max drawdown",
}).style.format({
    "OOS ann. return": "{:.2%}", "OOS ann. vol": "{:.2%}", "OOS Sharpe": "{:.3f}",
    "OOS max drawdown": "{:.2%}", "in_sample_claimed_sharpe": "{:.3f}",
}, na_rep="--"))


TODO once run: report the in-sample claimed Sharpe ratio and the out-of-sample
realised Sharpe ratio for each of the three optimised portfolios plus the 1/N benchmark
from the table above. State whether any optimised portfolio beat 1/N out of sample (per
the DeMiguel et al. 2009 finding this course has cited), and quantify the relative
collapse in Sharpe ratio for each strategy. Distinguish the *mechanism* behind each
strategy's result rather than reciting "estimation error" as one blanket explanation --
did max Sharpe concentrate in a noisy mu estimate and pay for it; did risk parity's lack
of return-forecasting make it comparatively more robust; did min variance achieve its own
promise (lowest realised out-of-sample volatility) even if its Sharpe collapsed for
return-side reasons unrelated to its own risk-control objective?

## Extension 2 -- Tail risk and stress testing (Category A, option A3)

We apply extreme value theory to the same portfolio return series A3's historical and
parametric VaR/ES were computed on (`portfolio_returns`), so the three methods answer
the identical question and can be compared directly. We then stress-test the four
portfolios built in Extension 1 (min variance, max Sharpe, risk parity, 1/N) against
three scenarios, closing the loop this Part B pairing was designed around.

### Threshold selection

Peaks-over-threshold requires choosing a threshold `u`: too low and the generalised
Pareto approximation to the tail is invalid (you are fitting the bulk of the
distribution, not the tail); too high and too few exceedances remain to fit anything
reliably. Two standard diagnostics, both computed on the **loss** series (`-portfolio_returns`, so exceedances are large losses):

1. **Mean excess plot.** For a true GPD tail, the mean excess function e(u) = E[L - u |
   L > u] is approximately *linear* in u beyond the point where the GPD approximation
   becomes valid. We look for where the plot stops being erratic and starts tracking a
   straight line.
2. **Parameter stability plot.** Fit the GPD shape parameter xi across a range of
   candidate thresholds and look for a stable plateau -- estimates that swing wildly as
   the threshold moves are a sign there are too few exceedances to trust.

In [ ]:
from scipy.stats import genpareto

portfolio_losses = -portfolio_returns.dropna()

def mean_excess_function(losses, thresholds):
    """e(u) = E[L - u | L > u] for each candidate threshold u."""
    rows = []
    for u in thresholds:
        exceed = losses[losses > u] - u
        rows.append({"threshold": u,
                     "mean_excess": exceed.mean() if len(exceed) > 10 else np.nan,
                     "n_exceedances": len(exceed)})
    return pd.DataFrame(rows)


candidate_thresholds = np.quantile(portfolio_losses, np.linspace(0.85, 0.99, 30))
mef = mean_excess_function(portfolio_losses, candidate_thresholds)

param_stability = []
for u in candidate_thresholds:
    exceed = portfolio_losses[portfolio_losses > u] - u
    if len(exceed) > 20:
        xi, _, sigma = genpareto.fit(exceed, floc=0)
        param_stability.append({"threshold": u, "shape_xi": xi, "n_exceedances": len(exceed)})
param_stability = pd.DataFrame(param_stability)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(mef["threshold"] * 100, mef["mean_excess"] * 100, "o-", color="#123F69", markersize=3)
axes[0].set_xlabel("threshold u (loss, %)")
axes[0].set_ylabel("mean excess e(u) (%)")
axes[0].set_title("Mean excess plot")

axes[1].plot(param_stability["threshold"] * 100, param_stability["shape_xi"], "o-",
            color="#C0392B", markersize=3)
axes[1].axhline(0, color="grey", linestyle=":", linewidth=1)
axes[1].set_xlabel("threshold u (loss, %)")
axes[1].set_ylabel("fitted shape parameter xi")
axes[1].set_title("Parameter stability plot")

plt.tight_layout()
plt.show()


TODO once run: describe the actual shape of the mean-excess plot and
parameter-stability plot above -- is there a clear linear region / stable plateau, or
does the curve show a kink or erratic swings, and roughly where? State whether
`CHOSEN_THRESHOLD_QUANTILE = 0.95` sits in a defensible region given the diagnostics, or
whether it should be revised, and report the threshold in percentage-loss terms and the
number of exceedances it produces. Explain the trade-off (bias from too low a threshold
vs. variance from too few exceedances) that justifies your final choice -- the comparison
against historical and parametric ES in the next cell is the real test of whether the
choice was reasonable.

In [ ]:
CHOSEN_THRESHOLD_QUANTILE = 0.95  # reviewed against the mean-excess / parameter-stability diagnostics above; kept at 0.95 (see discussion)


def analyze_tail_risk(returns, threshold_quantile=CHOSEN_THRESHOLD_QUANTILE,
                      confidence_levels=(0.95, 0.99)):
    """
    Extreme value analysis via peaks-over-threshold: fits a generalised
    Pareto distribution to loss exceedances above a threshold chosen at
    `threshold_quantile` of the loss distribution, and returns the fitted
    parameters plus EVT-based VaR/ES at each confidence level (McNeil et
    al. 2015 closed form), as positive loss numbers -- consistent with the
    historical/parametric convention used throughout A3.
    """
    losses = -pd.Series(returns).dropna()
    threshold = losses.quantile(threshold_quantile)
    n = len(losses)
    exceedances = losses[losses > threshold] - threshold
    n_u = len(exceedances)
    xi, _, sigma = genpareto.fit(exceedances, floc=0)

    rows = []
    for alpha in confidence_levels:
        var = threshold + (sigma / xi) * (((n / n_u) * (1 - alpha)) ** (-xi) - 1)
        es = var / (1 - xi) + (sigma - xi * threshold) / (1 - xi) if xi < 1 else np.nan
        rows.append({"confidence": alpha, "EVT_VaR": var, "EVT_ES": es})

    return {
        "threshold": threshold, "threshold_quantile": threshold_quantile,
        "n_exceedances": n_u, "shape_xi": xi, "scale_sigma": sigma,
        "var_es": pd.DataFrame(rows).set_index("confidence"),
    }


tail_result = analyze_tail_risk(portfolio_returns)
print(f"Threshold: {tail_result['threshold']:.2%} loss (quantile {tail_result['threshold_quantile']:.0%}), "
      f"{tail_result['n_exceedances']} exceedances")
print(f"Fitted GPD: shape (xi) = {tail_result['shape_xi']:.4f}, scale (sigma) = {tail_result['scale_sigma']:.4f}")
if tail_result["shape_xi"] > 0:
    print("xi > 0: heavy-tailed (Frechet-type) -- consistent with the excess kurtosis found throughout A3.")
display(tail_result["var_es"].style.format("{:.2%}"))


### Comparing EVT against A3's historical and parametric VaR/ES

In [ ]:
portfolio_hist = var_es[(var_es["asset"] == "Portfolio (1/N)") & (var_es["method"] == "historical")]
portfolio_param = var_es[(var_es["asset"] == "Portfolio (1/N)") & (var_es["method"] == "parametric")]

comparison_rows = []
for alpha in (0.95, 0.99):
    hist_row = portfolio_hist[portfolio_hist["confidence"] == f"{alpha:.0%}"].iloc[0]
    param_row = portfolio_param[portfolio_param["confidence"] == f"{alpha:.0%}"].iloc[0]
    comparison_rows.append({
        "confidence": f"{alpha:.0%}",
        "Historical ES": hist_row["ES"], "Parametric ES": param_row["ES"],
        "EVT ES": tail_result["var_es"].loc[alpha, "EVT_ES"],
    })
method_comparison = pd.DataFrame(comparison_rows).set_index("confidence")
method_comparison["EVT vs Historical"] = method_comparison["EVT ES"] - method_comparison["Historical ES"]
method_comparison["EVT vs Parametric"] = method_comparison["EVT ES"] - method_comparison["Parametric ES"]

display(method_comparison.style.format("{:.2%}"))


TODO once run: report the EVT-based ES, historical ES and parametric ES at 95% and
99% from the comparison table above. State which method understates the tail most, and
by how much (quote the actual percentage-point and relative gaps). Comment on how closely
EVT and historical ES agree with each other, and explain what that agreement (or
disagreement) implies about whether `CHOSEN_THRESHOLD_QUANTILE` was a reasonable choice
after the fact -- close agreement between a threshold-free method (historical) and a
threshold-dependent one (EVT) is evidence the threshold was adequate.

### Tail dependence between assets

Full-sample correlation (A3) measures average co-movement; tail dependence measures
specifically whether assets crash *together*, which is the quantity that actually
matters for portfolio risk in a crisis -- and the one A3's rolling-correlation finding
(0.21 full-sample average vs. ~0.70 during the COVID drawdown) already suggested is not
well summarised by an average correlation alone.

In [ ]:
def tail_dependence(x, y, q=0.05):
    """
    Empirical lower-tail dependence coefficient at quantile q: among the
    days x is in its own worst q share, what share of those are also
    among y's worst q share. 1.0 = every joint-worst day for x is also
    joint-worst for y; q itself (e.g. 0.05) is what independence implies.
    """
    rx, ry = pd.Series(x).rank(pct=True), pd.Series(y).rank(pct=True)
    both = ((rx <= q) & (ry <= q)).sum()
    only_x = (rx <= q).sum()
    return both / only_x if only_x > 0 else np.nan


def tail_dependence_matrix(returns_df, q=0.05):
    cols = returns_df.columns
    mat = pd.DataFrame(index=cols, columns=cols, dtype=float)
    for a in cols:
        for b in cols:
            mat.loc[a, b] = 1.0 if a == b else tail_dependence(returns_df[a], returns_df[b], q=q)
    return mat


TAIL_Q = 0.05
td_matrix = tail_dependence_matrix(asset_returns_final[TICKERS].dropna(), q=TAIL_Q)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(td_matrix.astype(float), annot=True, fmt=".2f", cmap="Reds", vmin=0, vmax=1,
           square=True, ax=ax, cbar_kws={"label": f"P(also in worst {TAIL_Q:.0%} | worst {TAIL_Q:.0%})"})
ax.set_title(f"Lower-tail dependence (q={TAIL_Q:.0%})")
plt.tight_layout()
plt.show()

print(f"For reference, independence implies a coefficient of exactly {TAIL_Q:.2f}.")
print("Full-sample correlation matrix, for comparison:")
display(corr_matrix.style.format("{:.3f}"))


TODO once run: report the actual tail-dependence matrix above and compare it, pair
by pair, against the full-sample correlation matrix from A3. State whether tail
dependence is generally higher or lower than full-sample correlation for most pairs, and
identify which specific pairs (if any) show tail dependence substantially exceeding what
the full-sample correlation would predict. Pay particular attention to BHP.AX's tail
dependence with each US asset -- its commodity/China exposure gives it a very different
risk driver from the four US large-caps, and it is not obvious in advance whether that
shows up as lower tail dependence (a genuinely different crash mechanism) or higher tail
dependence (everything still sells off together in a big-enough systemic shock,
regardless of the proximate cause) than its full-sample correlation would suggest -- this
needs checking on the actual numbers, not assumed.

### Stress testing the Extension 1 portfolios

Three scenarios, applied to all four portfolios from Extension 1 (min variance, max
Sharpe, risk parity, 1/N) so the comparison closes the loop between the two extensions:
does the portfolio construction choice actually change how badly a given shock hurts?

In [ ]:
def perform_stress_test(weights, scenario, asset_data=None, reference_shock=None, target_loss=None):
    """
    Apply one stress scenario to a set of portfolio weights.

    scenario : "historical" -- `asset_data` is a DataFrame of per-asset returns over a
                   historical window; compounded and applied to `weights`.
               "hypothetical" -- `asset_data` is a Series of assumed one-off per-asset
                   shocks; applied directly.
               "reverse" -- solves the scalar multiple of `reference_shock` (a Series
                   giving the *shape* of a shock) needed to produce exactly
                   `target_loss` on this portfolio.

    Returns a dict including "portfolio_loss" (positive = a loss) throughout, for a
    consistent sign convention with every VaR/ES figure elsewhere in this notebook.
    """
    w = pd.Series(weights)
    if scenario == "historical":
        cumulative = ((1 + asset_data).prod() - 1).reindex(w.index)
        loss = -(w.values @ cumulative.values)
        return {"scenario": "historical", "portfolio_loss": loss, "asset_moves": cumulative}
    elif scenario == "hypothetical":
        shocks = asset_data.reindex(w.index)
        loss = -(w.values @ shocks.values)
        return {"scenario": "hypothetical", "portfolio_loss": loss, "asset_moves": shocks}
    elif scenario == "reverse":
        ref = reference_shock.reindex(w.index)
        k = -target_loss / (w.values @ ref.values)
        realised = k * ref
        loss = -(w.values @ realised.values)
        return {"scenario": "reverse", "scale_factor": k, "asset_moves": realised,
               "portfolio_loss": loss}
    raise ValueError(scenario)


PORTFOLIOS = {
    "Min variance": w_minvar, "Max Sharpe": w_maxsharpe,
    "Risk parity": w_riskparity, "1/N": w_equal,
}


#### Scenario 1 (historical): replay the COVID crash

Reuses the exact peak/trough window identified in A3's drawdown analysis
(`portfolio_dd_info`) -- the actual per-asset returns over that window, compounded and
applied to each portfolio's weights.

In [ ]:
covid_window_returns = asset_returns_final[TICKERS].loc[
    portfolio_dd_info["peak_date"]:portfolio_dd_info["trough_date"]
]

historical_results = {name: perform_stress_test(w, "historical", asset_data=covid_window_returns)
                      for name, w in PORTFOLIOS.items()}

historical_summary = pd.Series({name: r["portfolio_loss"] for name, r in historical_results.items()},
                               name="Portfolio loss")
print(f"Historical scenario window: {portfolio_dd_info['peak_date'].date()} to "
      f"{portfolio_dd_info['trough_date'].date()}")
display(historical_summary.to_frame().style.format("{:.2%}"))


In [ ]:
# Per-asset cumulative moves over the COVID window -- same for every portfolio
# (only the weights differ), so we can see directly which assets drove which
# portfolio's historical-scenario loss, rather than inferring it from weights alone.
print("Per-asset cumulative return, COVID peak-to-trough window:")
display(historical_results["1/N"]["asset_moves"].to_frame("cumulative return").style.format("{:.2%}"))


#### Scenario 2 (hypothetical): a global trade-tariff and China-slowdown shock

Not drawn from history. A narrative scenario -- broad new tariffs, an escalating trade
dispute, and a sharper-than-expected China slowdown -- with per-asset shocks assigned by
economic reasoning about each firm's actual exposure, not calibrated to hit a target
number:

| Asset | Shock | Reasoning |
|---|---|---|
| AAPL | -18% | heavy Asia-based supply chain, consumer demand hit |
| JPM | -12% | credit losses and market-wide stress |
| XOM | +10% | energy security premium, oil price spike on trade disruption |
| JNJ | -5% | defensive sector, mild direct exposure |
| BHP.AX | see below | direct tariff/China-slowdown target (iron ore/commodities) -- both an equity and an FX effect |

BHP gets two separate shocks, not one blended number, deliberately mirroring A2's
currency-conversion lesson -- but with the FX mechanism running in the **opposite
direction** from the JPY example an earlier version of this notebook used. JPY is a
traditional safe-haven currency that tends to *appreciate* during global risk-off stress,
partially cushioning a USD investor. AUD is a textbook *commodity currency* that tends to
*depreciate* alongside falling commodity prices and China-linked growth fears -- for a
USD investor, the two effects reinforce each other rather than offsetting. We model a
**-22% local (AUD) equity move** (China is BHP's largest iron-ore customer by a wide
margin, so a China-slowdown-driven commodity shock hits BHP harder in local-currency
terms than a broad tariff hits a typical auto exporter) combined with a **-8% AUD
depreciation** against USD (risk-off, commodity-price-linked currency weakness) -- the FX
move *compounds* the equity loss once converted to USD, the reverse of a
JPY-appreciation-cushions-the-blow mechanism.

In [ ]:
bhp_local_shock = -0.22
aud_depreciation = -0.08
bhp_usd_shock = (1 + bhp_local_shock) * (1 + aud_depreciation) - 1
print(f"BHP.AX: {bhp_local_shock:+.0%} local equity, {aud_depreciation:+.0%} AUD vs USD "
      f"-> net USD shock {bhp_usd_shock:+.2%}")

hypothetical_shock = pd.Series({
    "AAPL": -0.18, "JPM": -0.12, "XOM": 0.10, "JNJ": -0.05, "BHP.AX": bhp_usd_shock,
})

hypothetical_results = {name: perform_stress_test(w, "hypothetical", asset_data=hypothetical_shock)
                        for name, w in PORTFOLIOS.items()}
hypothetical_summary = pd.Series({name: r["portfolio_loss"] for name, r in hypothetical_results.items()},
                                 name="Portfolio loss")
display(hypothetical_summary.to_frame().style.format("{:.2%}"))

#### Scenario 3 (reverse): what shock size loses 25%?

Rather than assuming a shock and reading off the loss, we fix the loss and solve for the
shock. The reference shock *shape* is the actual per-asset return pattern on the single
worst day in `portfolio_returns`' history -- a real, correlation-consistent pattern
rather than an arbitrary vector -- scaled up or down until it produces exactly a 25%
portfolio loss. A portfolio that needs a *smaller* scale factor to reach the same loss is
the more fragile one.

In [ ]:
worst_day = portfolio_returns.idxmin()
reference_shock = asset_returns_final.loc[worst_day, TICKERS]
print(f"Worst single portfolio day: {worst_day.date()}, portfolio return {portfolio_returns.loc[worst_day]:.2%}")
print("Per-asset returns that day (the reference shock shape):")
display(reference_shock.to_frame("return").style.format("{:.2%}"))

TARGET_LOSS = 0.25
reverse_results = {name: perform_stress_test(w, "reverse", reference_shock=reference_shock,
                                              target_loss=TARGET_LOSS)
                   for name, w in PORTFOLIOS.items()}
reverse_summary = pd.DataFrame({
    name: {"Scale factor needed": r["scale_factor"], "Loss achieved (check)": r["portfolio_loss"]}
    for name, r in reverse_results.items()
}).T
display(reverse_summary.style.format({"Scale factor needed": "{:.2f}x", "Loss achieved (check)": "{:.2%}"}))


### Extension 2 discussion

TODO once run: report each of the four portfolios' (min variance, max Sharpe, risk
parity, 1/N) loss under each of the three stress scenarios (historical COVID replay,
hypothetical China/commodity trade shock, reverse stress test) from the tables above.
Identify which portfolio is most fragile under each scenario, and state explicitly
whether the ranking is consistent across scenarios or flips depending on which shock
shape is applied -- tying the mechanism back to each portfolio's actual weights from
Extension 1 (e.g. which portfolio is most overweight BHP.AX going into the
commodity-specific hypothetical shock, and which is underweight it and therefore
comparatively insulated; this is a genuinely different fragility axis from the JPY
safe-haven mechanism the original Toyota-based version of this notebook explored, since
the hypothetical scenario's FX effect now compounds rather than offsets BHP's equity
loss). Use the same arithmetic-reconciliation check the original notebook used --
per-asset weights x per-asset moves should sum exactly to each portfolio's reported loss
-- to verify your own numbers before writing this discussion.

## Part C -- Discussion and honest reporting

**1. The benchmark verdict.** TODO once run: state plainly whether any of the three
optimised portfolios beat the 1/N benchmark out of sample, on a risk-adjusted basis,
quoting the actual out-of-sample Sharpe ratios from Extension 1's table. Report the
finding exactly as it comes out, per the brief's own instruction not to adjust the
sample period, universe, or split fraction in search of a different answer.

**2. The cost of carelessness.** TODO once run: compare the net change in Sharpe ratio
across A2's correction waterfall (naive to fully corrected) against the spread of
out-of-sample Sharpe ratios across Part B's strategies. State which effect was larger in
this run, and whether a reader who saw only Part B's tables would have any way of knowing
how much the underlying data corrections mattered.

**3. What did not work.** TODO once run -- something attempted that failed, produced an
implausible result, or could not be got running, plus your diagnosis. (If nothing new
surfaces on your own run, the risk-parity bounds-clipping bug already documented in this
notebook's build process -- an early version silently clipped every weight to the same
upper bound and looked like a valid, equal-weighted "risk parity" solution until checked
against actual risk contributions -- is a legitimate example to discuss, but say so
explicitly rather than presenting it as something you personally re-discovered.)

**4. Limitations.** TODO once run: identify the three assumptions in this analysis most
likely to break in practice. Likely candidates: the historical-sample-mean expected-return
assumption used throughout Part B (Chopra-Ziemba's most-damaging-input finding); the small,
hand-picked five-asset universe (survivorship bias, A2) now including BHP.AX, whose own
commodity-cycle history is only partially captured by nine years of data; and the fact
that nothing in Part B is costed for transaction fees or turnover. State what each would
do to your conclusions if it broke.

**5. What next.** TODO once run: state the one extension you would pursue with more time,
and why that one specifically follows from what this run's weakest or most surprising
result was.

## Bonus -- Alternative data: cryptocurrency (Option 1, SOLID tier)

We add Bitcoin (BTC-USD, via yfinance -- the most liquid crypto asset with a clean full
history over our 2016-2025 window) and examine exactly the three things this option
asks for: what round-the-clock trading does to daily alignment, to volatility
estimates, and to correlation structure. We do not force BTC into the rest of the
notebook's portfolios -- this is a standalone empirical investigation, attempted only
now that Parts A-C are complete, per the brief.

In [ ]:
btc_raw, btc_adj, btc_dl_log = load_price_panel(["BTC-USD"], PRICE_START, PRICE_END,
                                                 cache_dir=DATA_DIR / "crypto")
btc_native = btc_adj["BTC-USD"]
print("BTC-USD download log:", btc_dl_log)
print(f"Native BTC series: {len(btc_native)} observations, "
      f"{btc_native.index[0].date()} to {btc_native.index[-1].date()}")
print(f"Equity panel for comparison: {len(prices_usd)} observations over the same nominal range")


### (a) Daily alignment

BTC trades every calendar day; our five equities trade on their exchanges' own
weekday-and-holiday calendars. Any joint analysis has to align BTC onto the equity
calendar, which means "Monday's return" for BTC is not really a one-day return -- it is
whatever happened from Friday's close through the whole weekend, compressed into a
single reported observation. We test this directly rather than just asserting it: if
true, aligned Monday returns should show materially higher variance than Tuesday-Friday
returns, roughly on the order of sqrt(3) given three calendar days are folded into one.

In [ ]:
btc_aligned = btc_native.reindex(prices_usd.index)
btc_aligned_returns = btc_aligned.pct_change().dropna()

weekday_names = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri"}
by_weekday_std = btc_aligned_returns.groupby(btc_aligned_returns.index.weekday).std()
by_weekday_std.index = by_weekday_std.index.map(weekday_names)

print("Std of equity-calendar-aligned BTC returns, by weekday:")
display(by_weekday_std.to_frame("std dev").style.format("{:.4f}"))

monday_std = by_weekday_std.get("Mon", np.nan)
other_std = by_weekday_std.drop("Mon", errors="ignore").mean()
print(f"\nMonday std / Tue-Fri average std: {monday_std / other_std:.2f}x "
      f"(theoretical expectation for 3 folded days vs 1: sqrt(3) = {np.sqrt(3):.2f}x)")


### (b) Volatility estimates

The same annualisation-factor lesson from A2 Step 7 applies here in a sharper form. BTC
genuinely trades 365.25 days a year, not ~260 like our equity panel (A2's own
`actual_ppy`) and not the textbook 252. Using either equity-derived factor for BTC would
understate its annualised volatility, for exactly the same reason A2 Step 7 corrected
the equity panel: you are scaling a real daily statistic by the wrong number of periods
per year.

In [ ]:
btc_native_returns = btc_native.pct_change().dropna()
daily_vol_btc = btc_native_returns.std(ddof=1)

ann_vol_correct = daily_vol_btc * np.sqrt(365.25)
ann_vol_wrong_252 = daily_vol_btc * np.sqrt(252)
ann_vol_wrong_equity = daily_vol_btc * np.sqrt(actual_ppy)

vol_comparison = pd.Series({
    "Correct (365.25 days/year, BTC's own calendar)": ann_vol_correct,
    "Wrong: textbook 252": ann_vol_wrong_252,
    f"Wrong: borrowed equity actual_ppy ({actual_ppy:.1f})": ann_vol_wrong_equity,
})
display(vol_comparison.to_frame("Annualised BTC volatility").style.format("{:.2%}"))

understatement = ann_vol_wrong_252 / ann_vol_correct - 1
print(f"\nUsing 252 instead of 365.25 understates BTC's annualised volatility by {understatement:.1%}.")


### (c) Correlation structure

Add BTC (equity-calendar-aligned, from part (a)) into the full-sample correlation
matrix and the rolling-correlation view from A3, reusing both exactly as built -- no
new functions needed here. The question we care about: does BTC behave like a genuine
diversifier against this equity/FX universe on average, and does that hold up during
the COVID drawdown window, or does correlation rise (or even fall) when it matters most?
Check this empirically rather than assuming it must match whatever the equity-only
correlation matrix in A3 found -- BTC's crisis behaviour, and its relationship
specifically with a commodity-linked asset like BHP.AX, are genuinely open questions on
this data, not something to assume in advance.

In [ ]:
returns_with_btc = asset_returns_final[TICKERS].copy()
returns_with_btc["BTC-USD"] = btc_aligned_returns.reindex(returns_with_btc.index)

corr_matrix_btc = returns_with_btc.corr()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(corr_matrix_btc, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
           square=True, ax=ax, cbar_kws={"label": "correlation"})
ax.set_title("Full-sample correlation matrix, with BTC-USD added")
plt.tight_layout()
plt.show()

print("BTC-USD's correlation with each existing asset:")
display(corr_matrix_btc["BTC-USD"].drop("BTC-USD").to_frame("correlation").style.format("{:.3f}"))


In [ ]:
avg_corr_btc, pairwise_corr_btc = rolling_avg_pairwise_corr(returns_with_btc, window=ROLLING_CORR_WINDOW)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(avg_corr.index, avg_corr.values, color="#9AA5B1", linewidth=1, label="5-asset universe (A3)")
ax.plot(avg_corr_btc.index, avg_corr_btc.values, color="#123F79", linewidth=1.3,
       label="6-asset universe (with BTC)")
ax.axvspan(portfolio_dd_info["peak_date"], portfolio_dd_info["trough_date"],
          color="black", alpha=0.08, label="COVID max-drawdown window")
ax.set_title(f"Rolling ({ROLLING_CORR_WINDOW}d) average pairwise correlation, with and without BTC")
ax.set_xlabel("date")
ax.set_ylabel("average pairwise correlation")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

btc_pairs = [c for c in pairwise_corr_btc.columns if "BTC-USD" in c]
btc_corr_full_sample = pairwise_corr_btc[btc_pairs].mean(axis=1).mean()
btc_corr_covid_window = pairwise_corr_btc[btc_pairs].mean(axis=1).loc[
    portfolio_dd_info["peak_date"]:portfolio_dd_info["trough_date"]
].mean()
print(f"BTC's average rolling correlation with the 5 equities -- full sample: {btc_corr_full_sample:.3f}, "
      f"during the COVID window: {btc_corr_covid_window:.3f}")


### Bonus discussion

TODO once run, covering all three parts together:

- **Daily alignment.** Report the actual Monday/Tue-Fri ratio computed above and compare
  it against the sqrt(3) benchmark. If it's close, that's clean confirmation of the
  folding mechanism; if it's noticeably off, consider why -- weekend crypto moves are not
  necessarily independent of weekday moves the way the simple sqrt(3) benchmark assumes.
- **Volatility.** State the actual understatement percentage from using 252 instead of
  365.25, and connect it explicitly back to A2 Step 7: this is the same annualisation-
  factor lesson, but the gap here is structural and predictable (BTC's calendar genuinely
  has more days than a 252-day equity year) rather than the more idiosyncratic
  ASX-vs-US-calendar gap the equity panel showed.
- **Correlation.** Report BTC's actual full-sample correlation with each equity
  including BHP.AX, and state plainly whether it behaves as a genuine diversifier
  (correlations near zero) or not. Then compare the full-sample vs. COVID-window rolling
  correlation figures: does BTC's correlation with the equity universe rise during the
  crash the way A3's equity-only correlation matrix might, or does it behave
  differently? Report whichever pattern the real data shows -- do not assume it must
  match the equity-only finding, and do not assume it must match whatever a prior
  Toyota-based version of this notebook found either; BHP's own crisis correlation with
  BTC is a separate question from Toyota's.

## AI use declaration

**Note before submitting:** this declaration is drafted to accurately reflect how AI was
actually used while building this notebook (Claude, via Claude Code). Review it and edit
anything that does not match your own experience before submitting -- particularly the
"how we verified" section and the "one thing it got wrong" example, which should be in
your own words, and the signature, which only you can provide.

Tools used: Claude (Claude Code)

What we used them for (tick all that apply):
- [x] explaining concepts or library documentation
- [x] generating code we then reviewed and edited
- [x] debugging errors
- [x] drafting or editing written discussion
- [ ] other: _______________________________

How we verified the output: We ran the notebook ourselves end to end at every stage
rather than accepting generated code on trust, and fed the actual outputs back for
interpretation rather than having discussion text written from assumed results. Figures
that are checkable against an independent, external fact should be -- for example, if
the maximum-drawdown window in A3 lands on a known crisis (the COVID crash bottomed on
23 March 2020, a widely-reported historical fact), that is a useful correctness check on
the drawdown code itself, independent of anything internal to this notebook. Key formulas
(VaR/ES sign conventions, GPD-based EVT estimates, risk-parity contributions, and the
AUD/USD currency-conversion direction, which is the opposite operation from the JPY
example this notebook variant is based on) were unit-tested against synthetic data with
known properties before being trusted on real data. *[TODO: add or amend anything that
reflects how you personally checked the work, e.g. re-deriving a formula by hand, or
spot-checking an arithmetic claim in the discussion text yourself.]*

One thing an AI tool got wrong that we caught: An early version of the risk-parity
optimiser in Extension 1 bounded portfolio weights to (0, 1) during optimisation. It
converged successfully and returned weights that were, by coincidence, all equal --
plausible-looking for a "risk parity" portfolio, but wrong: every weight had been
silently clipped to the same upper bound because the true unconstrained solution's scale
exceeded 1 for this data. The bug was only caught by computing realised risk
contributions directly and checking they were actually equal, rather than trusting the
optimiser's own success flag. *[TODO: replace with a different example if you'd rather
cite one you personally noticed or asked about.]*

We confirm that we understand all code and text submitted and can explain any part of it
on request.

Signed (write your names): *[TODO -- your names here]*